# Bibliotecas

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime
import itertools

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.colors as pc
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifierCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score, make_scorer, f1_score, precision_score, recall_score, precision_recall_curve, average_precision_score, roc_curve, auc, fbeta_score
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_validate, StratifiedGroupKFold, StratifiedShuffleSplit, GridSearchCV
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC, OneClassSVM
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.feature_selection import SelectKBest, f_classif

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from scipy.signal import find_peaks
from scipy.integrate import trapezoid
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import skew, kurtosis, gaussian_kde, mannwhitneyu, ks_2samp, pearsonr, spearmanr, kendalltau, kruskal
from scipy import stats

from hampel import hampel

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd()+"/data/"
DIR_OUTPUT = os.getcwd()+"/output/"

# Carregando dados

## Crystallizer #1

In [ ]:
base_name_crystallizer1 = "Crystallizer #1.csv"

df_crystallizer1 = pd.read_csv(DIR_DATA + base_name_crystallizer1, sep=";", decimal=".")
df_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer1["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer1["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer1.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer1 = df_crystallizer1[df_crystallizer1.duplicated(subset=['Labref'], keep=False)]

# Retirando linhas 23 e 2141 que estão duplicadas mas não possuem amostras significativas (i.e amostras com valores muito baixos)
df_crystallizer1.drop([23,2141], inplace=True) 
# Aplicando média para medidas com Labref iguais (apenas as que possuem valores próximos)
agg_logic = {col: 'mean' if df_crystallizer1[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer1.columns if col != 'Labref'}
df_crystallizer1 = df_crystallizer1.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer1 = df_crystallizer1.drop(remove.index)
df_crystallizer1

### Removendo outliers 0 a 10 #1

In [ ]:
df_crystallizer1_0a10 = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer1)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer1_0a10)} ({len(df_crystallizer1) - len(df_crystallizer1_0a10)} removidas)")

df_crystallizer1_0a10

### Removendo outliers com IQR #1

In [ ]:
serie = df_crystallizer1["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer1_iqr = df_crystallizer1[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer1)}")
print(f"Amostras removidas: {len(df_crystallizer1) - len(df_crystallizer1_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer1_iqr)}")

df_crystallizer1_iqr

### Removendo outliers com Filtro de Hampel #1

In [ ]:
serie = df_crystallizer1["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer1_hampel = df_crystallizer1.copy().reset_index(drop=True)
df_crystallizer1_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer1_hampel

## Crystallizer #2

In [ ]:
base_name_crystallizer2 = "Crystallizer #2.csv"

df_crystallizer2 = pd.read_csv(DIR_DATA + base_name_crystallizer2, sep=";", decimal=".")
df_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer2["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer2["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer2.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer2 = df_crystallizer2[df_crystallizer2.duplicated(subset=['Labref'], keep=False)]

# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer2[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer2.columns if col != 'Labref'}
df_crystallizer2 = df_crystallizer2.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostra 4027521 que possui valor muito discrepante
idx = df_crystallizer2[df_crystallizer2['Labref'] == 4027521].index 
df_crystallizer2 = df_crystallizer2.drop(idx)


# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer2 = df_crystallizer2.drop(remove.index)
df_crystallizer2

### Removendo outliers 0 a 10 #2

In [ ]:
df_crystallizer2_0a10 = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer2)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer2_0a10)} ({len(df_crystallizer2) - len(df_crystallizer2_0a10)} removidas)")

df_crystallizer2_0a10

### Removendo outliers com IQR #2

In [ ]:
serie = df_crystallizer2["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer2_iqr = df_crystallizer2[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer2)}")
print(f"Amostras removidas: {len(df_crystallizer2) - len(df_crystallizer2_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer2_iqr)}")

df_crystallizer2_iqr

### Removendo outliers com Filtro de Hampel #2

In [ ]:
serie = df_crystallizer2["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer2_hampel = df_crystallizer2.copy().reset_index(drop=True)
df_crystallizer2_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer2_hampel

## Crystallizer #3

In [ ]:
base_name_crystallizer3 = "Crystallizer #3.csv"

df_crystallizer3 = pd.read_csv(DIR_DATA + base_name_crystallizer3, sep=";", decimal=".")
df_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer3["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer3["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer3.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer3 = df_crystallizer3[df_crystallizer3.duplicated(subset=['Labref'], keep=False)]


# Retirando linhas 6476 e 6494 que estão duplicadas mas não possuem amostras significativas
df_crystallizer3.drop([6476,6494], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer3[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer3.columns if col != 'Labref'}
df_crystallizer3 = df_crystallizer3.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer3 = df_crystallizer3.drop(remove.index)
df_crystallizer3

### Removendo outliers 0 a 10 #3

In [ ]:
df_crystallizer3_0a10 = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer3)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer3_0a10)} ({len(df_crystallizer3) - len(df_crystallizer3_0a10)} removidas)")

df_crystallizer3_0a10

### Removendo outliers com IQR #3

In [ ]:
serie = df_crystallizer3["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer3_iqr = df_crystallizer3[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer3)}")
print(f"Amostras removidas: {len(df_crystallizer3) - len(df_crystallizer3_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer3_iqr)}")

df_crystallizer3_iqr

### Removendo outliers com Filtro de Hampel #3

In [ ]:
serie = df_crystallizer3["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer3_hampel = df_crystallizer3.copy().reset_index(drop=True)
df_crystallizer3_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer3_hampel

# Carregando eventos identificados

## Deifinindo LC

In [ ]:
threshold = 5

## Crystallizer #1

In [ ]:
base_name_eventos_crystallizer1 = "Eventos-Reator1.csv"

df_eventos_crystallizer1 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer1, sep=";", decimal=".")
df_eventos_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer1["Real"] = 1

nova_linha = {
    "TIMESTAMP": pd.to_datetime("2025-10-21 08:38:00"),
    "Real": 0,
    "Evento": "Falso Alarme"
}

df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, pd.DataFrame([nova_linha])], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)

## Removendo eventos que não são trocas de reator
# df_eventos_crystallizer1.drop([2,3,4,6,7,8], inplace=True) 
df_eventos_crystallizer1

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer1 original
    diffs = (df_eventos_crystallizer1["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer1 existente
df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer1

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer1_filtrado_until2020 = df_eventos_crystallizer1[df_eventos_crystallizer1["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer1_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer1_filtrado_after2020 = df_eventos_crystallizer1[df_eventos_crystallizer1["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer1_filtrado_after2020.head()

## Crystallizer #2

In [ ]:
base_name_eventos_crystallizer2 = "Eventos-Reator2.csv"

df_eventos_crystallizer2 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer2, sep=";", decimal=".")
df_eventos_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer2["Real"] = 1

# Removendo eventos fora do período de dados
df_eventos_crystallizer2.drop([0,1,2], inplace=True) 

## Removendo eventos que não são trocas de reator
# df_eventos_crystallizer2.drop([6,9], inplace=True) 
df_eventos_crystallizer2

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer2 original
    diffs = (df_eventos_crystallizer2["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer2 existente
df_eventos_crystallizer2 = pd.concat([df_eventos_crystallizer2, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer2 = df_eventos_crystallizer2.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer2

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer2_filtrado_until2020 = df_eventos_crystallizer2[df_eventos_crystallizer2["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer2_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer2_filtrado_after2020 = df_eventos_crystallizer2[df_eventos_crystallizer2["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer2_filtrado_after2020.head()

## Crystallizer #3

In [ ]:
base_name_eventos_crystallizer3 = "Eventos-Reator3.csv"

df_eventos_crystallizer3 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer3, sep=";", decimal=".")
df_eventos_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer3["Real"] = 1

df_eventos_crystallizer3.drop([0,1], inplace=True) # Removendo eventos fora do período de dados

## Removendo eventos que não são trocas de reator
# df_eventos_crystallizer3.drop([4,7], inplace=True) 
df_eventos_crystallizer3

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer3 original
    diffs = (df_eventos_crystallizer3["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer3 existente
df_eventos_crystallizer3 = pd.concat([df_eventos_crystallizer3, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer3 = df_eventos_crystallizer3.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer3

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer3_filtrado_until2020 = df_eventos_crystallizer3[df_eventos_crystallizer3["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer3_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer3_filtrado_after2020 = df_eventos_crystallizer3[df_eventos_crystallizer3["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer3_filtrado_after2020.head()

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

In [ ]:
def plot_crystallizer(df_med, df_eventos, titulo, mostrar_falsos=True):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_med['TIMESTAMP'],
        y=df_med["Resultado de Ferro (ppm)"],
        mode='lines',
        name="Resultado de Ferro (ppm)",
        line=dict(color='black')
    ))
    fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

    for _, row in df_eventos.iterrows():
        if not mostrar_falsos and row["Real"] == 0:
            continue

        cor = "red" if row["Real"] == 1 else "blue"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0, y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

    fig.update_layout(
        template='plotly_white',
        hovermode='x unified',
        title=titulo
    )
    return fig

## Crystallizer #1

In [ ]:
# Sem eventos falsos
fig_crystallizer1 = plot_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer1.show()

In [ ]:
# fig_crystallizer1.write_html("Crystallizer#1.html")

### Crystallizer #1 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer1_0a10 = plot_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 0a10 (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer1_0a10.show()

### Crystallizer #1 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer1_iqr = plot_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 IQR (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer1_iqr.show()

### Crystallizer #1 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer1_hampel = plot_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 Hampel (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer1_hampel.show()

## Crystallizer #2

In [ ]:
# Sem eventos falsos
fig_crystallizer2 = plot_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer2.show()

In [ ]:
# fig_crystallizer2.write_html("Crystallizer#2.html")

### Crystallizer #2 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer2_0a10 = plot_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 0a10 (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer2_0a10.show()

### Crystallizer #2 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer2_iqr = plot_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 IQR (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer2_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer2_hampel = plot_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 Hampel (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer2_hampel.show()

## Crystallizer #3

In [ ]:
# Sem eventos falsos
fig_crystallizer3 = plot_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer3.show()

In [ ]:
# fig_crystallizer3.write_html("Crystallizer#3.html")

### Crystallizer #3 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer3_0a10 = plot_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 0a10 (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer3_0a10.show()

### Crystallizer #3 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer3_iqr = plot_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 IQR (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer3_iqr.show()

### Crystallizer #3 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer3_hampel = plot_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 Hampel (somente eventos reais)",
    mostrar_falsos=False
)
fig_crystallizer3_hampel.show()

## Crystallizer #1#2#3

In [ ]:
def plot_crystallizers(crystallizers, titulo, mostrar_falsos=True):
    fig = go.Figure()

    for c in crystallizers:
        fig.add_trace(go.Scatter(
            x=c["df"]['TIMESTAMP'],
            y=c["df"]["Resultado de Ferro (ppm)"],
            mode='lines',
            name=c["nome"],
            line=dict(color=c["cor"])
        ))

        for _, row in c["df_eventos"].iterrows():
            if not mostrar_falsos and row["Real"] == 0:
                continue

            cor = c["cor"] if row["Real"] == 1 else "gray"
            fig.add_shape(
                type="line",
                x0=str(row["TIMESTAMP"]),
                x1=str(row["TIMESTAMP"]),
                y0=0, y1=1,
                yref="paper",
                line=dict(color=cor, width=1.5, dash="dash")
            )
            fig.add_annotation(
                x=str(row["TIMESTAMP"]),
                y=1,
                yref="paper",
                text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
                showarrow=False,
                textangle=-90,
                yanchor="top",
                font=dict(color=cor)
            )

    fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")
    fig.update_layout(
        template='plotly_white',
        hovermode='x unified',
        title=titulo
    )
    return fig

In [ ]:
crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3 (somente eventos reais)", mostrar_falsos=False)
fig.show()

### Outliers 0 a 10

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_0a10, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_0a10, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_0a10, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3 0a10 (somente eventos reais)", mostrar_falsos=False)
fig.show()

### Outliers IQR

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_iqr, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_iqr, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_iqr, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3 IQR (somente eventos reais)", mostrar_falsos=False)
fig.show()

### Outliers Hampel

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_hampel, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_hampel, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_hampel, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3 Hampel (somente eventos reais)", mostrar_falsos=False)
fig.show()

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

In [ ]:
def plot_mm_crystallizer(df_med, df_eventos, titulo, janelas=None, mostrar_falsos=True):
    if janelas is None:
        janelas = [15, 12, 9, 6, 3]
    cores_mm = ['blue', 'green', 'orange', 'purple', 'red']

    df_mm = df_med.copy().set_index('TIMESTAMP')
    for dias in janelas:
        df_mm[f'MM_{dias}D'] = df_mm["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()
    df_mm = df_mm.reset_index()

    fig = go.Figure()

    # Série original
    fig.add_trace(go.Scatter(
        x=df_mm['TIMESTAMP'],
        y=df_mm["Resultado de Ferro (ppm)"],
        mode='lines',
        name="Resultado de Ferro (ppm)",
        line=dict(color='gray', width=1),
        opacity=0.6
    ))

    # Médias móveis
    for dias, cor in zip(janelas, cores_mm):
        fig.add_trace(go.Scatter(
            x=df_mm['TIMESTAMP'],
            y=df_mm[f'MM_{dias}D'],
            mode='lines',
            name=f"MM {dias}D",
            line=dict(color=cor, width=2)
        ))

    fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

    for _, row in df_eventos.iterrows():
        if not mostrar_falsos and row["Real"] == 0:
            continue

        cor = "red" if row["Real"] == 1 else "blue"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0, y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top"
        )

    fig.update_layout(
        template='plotly_white',
        hovermode='x unified',
        title=titulo
    )
    return fig

## MM Crystallizer #1

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 0a10 (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 IQR (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 Hampel (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #2

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 0a10 (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 IQR (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 Hampel (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #3

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 0a10 (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 IQR (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 Hampel (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #1#2#3

In [ ]:
def plot_mm_crystallizers(crystallizers, num_dias=7, titulo=None, mostrar_falsos=True):
    CORES = {
        "Crystallizer #1": "blue",
        "Crystallizer #2": "green",
        "Crystallizer #3": "orange"
    }

    fig = go.Figure()

    for c in crystallizers:
        cor = CORES[c["nome"]]

        df_mm = c["df"].copy().set_index('TIMESTAMP')
        df_mm[f'MM_{num_dias}D'] = df_mm["Resultado de Ferro (ppm)"].rolling(window=f'{num_dias}D').mean()
        df_mm = df_mm.reset_index()

        fig.add_trace(go.Scatter(
            x=df_mm['TIMESTAMP'],
            y=df_mm["Resultado de Ferro (ppm)"],
            mode='lines',
            name=f"{c['nome']} — original",
            line=dict(color=cor, width=1),
            opacity=0.3
        ))

        fig.add_trace(go.Scatter(
            x=df_mm['TIMESTAMP'],
            y=df_mm[f'MM_{num_dias}D'],
            mode='lines',
            name=f"{c['nome']} — MM {num_dias}D",
            line=dict(color=cor, width=2)
        ))

        for _, row in c["df_eventos"].iterrows():
            if not mostrar_falsos and row["Real"] == 0:
                continue

            dash_evento = "solid" if row["Real"] == 1 else "dash"
            fig.add_shape(
                type="line",
                x0=str(row["TIMESTAMP"]),
                x1=str(row["TIMESTAMP"]),
                y0=0, y1=1,
                yref="paper",
                line=dict(color=cor, width=1.5, dash=dash_evento)
            )
            fig.add_annotation(
                x=str(row["TIMESTAMP"]),
                y=1,
                yref="paper",
                text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
                showarrow=False,
                textangle=-90,
                yanchor="top",
                font=dict(color=cor)
            )

    fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")
    fig.update_layout(
        template='plotly_white',
        hovermode='x unified',
        title=titulo or f"Fe (ppm) MM {num_dias}D — Crystallizers #1, #2 e #3"
    )
    return fig

In [ ]:
crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3"},
]

# Sem eventos falsos
fig = plot_mm_crystallizers(crystallizers, num_dias=7, mostrar_falsos=False)
fig.show()

# Estatísticas descritivas de toda série de concentração de Fe

In [ ]:
def estatisticas(serie, nome):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    return pd.Series({
        "Contagem"     : serie.count(),
        "Média"        : serie.mean(),
        "Mediana"      : serie.median(),
        "Desvio Padrão": serie.std(),
        "Variância"    : serie.var(),
        "Mínimo"       : serie.min(),
        "Máximo"       : serie.max(),
        "Amplitude"    : serie.max() - serie.min(),
        "Q1 (25%)"     : Q1,
        "Q3 (75%)"     : Q3,
        "IQR"          : Q3 - Q1,
        "Assimetria"   : serie.skew(),
        "Curtose"      : serie.kurt()
    }, name=nome)

# Coluna separadora vazia
separador = pd.Series({k: "" for k in ["Contagem","Média","Mediana","Desvio Padrão","Variância",
                                        "Mínimo","Máximo","Amplitude","Q1 (25%)","Q3 (75%)","IQR",
                                        "Assimetria","Curtose"]})

c1 = pd.concat([
    estatisticas(df_crystallizer1["Resultado de Ferro (ppm)"].dropna(),       "C1 Original"),
    estatisticas(df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna(),  "C1 Intervalo 0-10"),
    estatisticas(df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna(),   "C1 IQR"),
    estatisticas(df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna(),"C1 Hampel"),
], axis=1)

c2 = pd.concat([
    estatisticas(df_crystallizer2["Resultado de Ferro (ppm)"].dropna(),       "C2 Original"),
    estatisticas(df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna(),  "C2 Intervalo 0-10"),
    estatisticas(df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna(),   "C2 IQR"),
    estatisticas(df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna(),"C2 Hampel"),
], axis=1)

c3 = pd.concat([
    estatisticas(df_crystallizer3["Resultado de Ferro (ppm)"].dropna(),       "C3 Original"),
    estatisticas(df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna(),  "C3 Intervalo 0-10"),
    estatisticas(df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna(),   "C3 IQR"),
    estatisticas(df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna(),"C3 Hampel"),
], axis=1)

sep = separador.rename("│")

df_comparativo = pd.concat([c1, sep, c2, sep.rename("│"), c3], axis=1).round(4)

# Corrige as colunas separadoras que ficaram com float após o round
df_comparativo["│"]  = ""
df_comparativo["│"] = ""

df_comparativo

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

CORES = {"C1": "#1f77b4", "C2": "#2ca02c", "C3": "#ff7f0e"}

def hex_to_rgba(cor, alpha=0.15):
    rgb = pc.hex_to_rgb(cor)
    return f'rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})'

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=2,
    subplot_titles=[
        titulo
        for m in metodos
        for titulo in [f"KDE — {m['titulo']}", f"Violin Plot — {m['titulo']}"]
    ]
)

for row_idx, metodo in enumerate(metodos, start=1):
    for c in metodo["series"]:
        serie = c["serie"]
        nome  = c["nome"]
        cor   = CORES[nome]

        # KDE na escala de densidade natural (área sob a curva = 1)
        kde = gaussian_kde(serie)
        x_range = np.linspace(serie.min(), serie.max(), 1000)
        y_kde = kde(x_range)
        y_kde = y_kde / trapezoid(y_kde, x_range)  # normaliza

        fig.add_trace(go.Scatter(
            x=x_range,
            y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=hex_to_rgba(cor, alpha=0.15),
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Violin
        fig.add_trace(go.Violin(
            y=serie,
            name=nome,
            marker_color=cor,
            fillcolor=hex_to_rgba(cor, alpha=0.4),
            box_visible=True,
            meanline_visible=True,
            legendgroup=nome,
            showlegend=False
        ), row=row_idx, col=2)

    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)
    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_yaxes(title_text="ppm", row=row_idx, col=2)

fig.update_layout(
    height=500 * n_metodos,
    template='plotly_white',
    title="Análise Descritiva Comparativa — Resultado de Ferro (ppm)",
)
fig.show()

### Teste de Kruskal-Wallis com Effect Size (η²)

Para amostras grandes (n ~ 30.000), testes estatísticos como o Kruskal-Wallis tendem a rejeitar a hipótese nula mesmo com diferenças praticamente irrelevantes. Por isso o p-value é complementado pelo **eta-quadrado (η²)** que mede a proporção da variação total explicada pelo agrupamento por crystallizer.

| η²        | Interpretação                                      |
|-----------|----------------------------------------------------|
| < 0.01    | Efeito negligenciável — unificação justificada     |
| 0.01–0.06 | Efeito pequeno — unificação provavelmente aceitável|
| 0.06–0.14 | Efeito médio — avaliar com cautela                 |
| > 0.14    | Efeito grande — distribuições substancialmente diferentes |

A decisão de unificar os dados dos três crystallizers deve considerar em conjunto o η², o p-value e a inspeção visual das curvas KDE e violin plots gerados.

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

for metodo in metodos:
    series = [c["serie"].values for c in metodo["series"]]
    nomes  = [c["nome"] for c in metodo["series"]]
    n_total = sum(len(s) for s in series)

    stat_kw, p_kw = kruskal(*series)

    # Eta-quadrado: mede o quanto da variação total é explicada pelo grupo
    # 0.01 = pequeno, 0.06 = médio, 0.14 = grande
    eta2 = (stat_kw - len(series) + 1) / (n_total - len(series))

    print(f"\nMétodo: {metodo['titulo']}")
    print(f"  H = {stat_kw:.4f}  |  p = {p_kw:.6f}  |  η² = {eta2:.4f}")
    if eta2 < 0.01:
        print("  → Efeito negligenciável — unificação justificada mesmo com p < 0.05")
    elif eta2 < 0.06:
        print("  → Efeito pequeno — unificação provavelmente aceitável")
    elif eta2 < 0.14:
        print("  → Efeito médio — avaliar com cautela")
    else:
        print("  → Efeito grande — distribuições substancialmente diferentes")

### Teste de normalidade Q-Q Plot

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

serie = df["Resultado de Ferro (ppm)"].dropna()

# Visualização
fig_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie.min(), serie.max(), 300)
y_normal = stats.norm.pdf(x_range, serie.mean(), serie.std())
y_normal_scaled = y_normal * len(serie) * (serie.max() - serie.min()) / 50

fig_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_normalidade_crystallizer1.show()

# Unificando bases de dados

## Unificando dados

In [ ]:
# Original
df_crystallizer123 = pd.concat([
    df_crystallizer1.assign(Crystallizer='C1'),
    df_crystallizer2.assign(Crystallizer='C2'),
    df_crystallizer3.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Originais: {df_crystallizer123.shape}")

# Intervalo 0-10
df_crystallizer123_0a10 = pd.concat([
    df_crystallizer1_0a10.assign(Crystallizer='C1'),
    df_crystallizer2_0a10.assign(Crystallizer='C2'),
    df_crystallizer3_0a10.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Intervalo 0-10: {df_crystallizer123_0a10.shape}")

# IQR
df_crystallizer123_iqr = pd.concat([
    df_crystallizer1_iqr.assign(Crystallizer='C1'),
    df_crystallizer2_iqr.assign(Crystallizer='C2'),
    df_crystallizer3_iqr.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"IQR: {df_crystallizer123_iqr.shape}")

# Hampel
df_crystallizer123_hampel = pd.concat([
    df_crystallizer1_hampel.assign(Crystallizer='C1'),
    df_crystallizer2_hampel.assign(Crystallizer='C2'),
    df_crystallizer3_hampel.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Hampel: {df_crystallizer123_hampel.shape}")

## Unificando eventos

In [ ]:
# Eventos unificados — igual para todos os tratamentos
df_eventos_crystallizer123 = pd.concat([
    df_eventos_crystallizer1.assign(Crystallizer='C1'),
    df_eventos_crystallizer2.assign(Crystallizer='C2'),
    df_eventos_crystallizer3.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')

INTERVALO_MIN_DIAS = 15

# =============================================================================
# ETAPA 1 — Remover Real=0 que estejam a menos de INTERVALO_MIN_DIAS de qualquer evento Real=1
# =============================================================================

timestamps_real1 = df_eventos_crystallizer123.loc[
    df_eventos_crystallizer123['Real'] == 1, 'TIMESTAMP'
]

def proximo_de_real1(ts, timestamps_real1, intervalo_dias):
    """Retorna True se ts está dentro do intervalo de algum Real=1."""
    diffs = (timestamps_real1 - ts).abs()
    return (diffs <= pd.Timedelta(days=intervalo_dias)).any()

mask_real0_proximo = df_eventos_crystallizer123['Real'] == 0
mask_real0_proximo = mask_real0_proximo & df_eventos_crystallizer123['TIMESTAMP'].apply(
    lambda ts: proximo_de_real1(ts, timestamps_real1, INTERVALO_MIN_DIAS)
)

removidos_por_real1 = mask_real0_proximo.sum()
df_eventos_crystallizer123 = df_eventos_crystallizer123[~mask_real0_proximo].copy()

print(f"Real=0 removidos por proximidade a Real=1: {removidos_por_real1}")

# =============================================================================
# ETAPA 2 — Retirar Real=0 restantes com intervalo <= INTERVALO_MIN_DIAS mantendo o que vem primeiro cronologicamente
# =============================================================================

df_real1    = df_eventos_crystallizer123[df_eventos_crystallizer123['Real'] == 1].copy()
df_real0    = df_eventos_crystallizer123[df_eventos_crystallizer123['Real'] == 0] \
                .sort_values('TIMESTAMP').copy()

real0_filtrados = []
ultimo_ts       = None

for _, row in df_real0.iterrows():
    ts_atual = row['TIMESTAMP']
    if ultimo_ts is None or (ts_atual - ultimo_ts).days > INTERVALO_MIN_DIAS:
        real0_filtrados.append(row)
        ultimo_ts = ts_atual

df_real0_limpo = pd.DataFrame(real0_filtrados)

print(f"\nReal=0 antes da deduplicação  : {len(df_real0)}")
print(f"Real=0 após a deduplicação    : {len(df_real0_limpo)}")
print(f"Removidos na deduplicação     : {len(df_real0) - len(df_real0_limpo)}")

df_eventos_crystallizer123 = pd.concat(
    [df_real1, df_real0_limpo], ignore_index=True
).sort_values('TIMESTAMP').reset_index(drop=True)

print(f"\nDistribuição final:")
print(df_eventos_crystallizer123['Real'].value_counts().to_string())
print(f"Total de eventos: {len(df_eventos_crystallizer123)}")

## Violin Plot das classes

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer123},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer123_0a10},
    {"titulo": "IQR",              "df": df_crystallizer123_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer123_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer123.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1#2#3 unificado"
)
fig.show()

## Plot dados unificados

In [ ]:
def plot_crystallizer_unificado(df_med, df_eventos, titulo, mostrar_falsos=True):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_med['TIMESTAMP'],
        y=df_med["Resultado de Ferro (ppm)"],
        mode='lines',
        name="Resultado de Ferro (ppm)",
        line=dict(color='black')
    ))

    fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

    for _, row in df_eventos.iterrows():
        if not mostrar_falsos and row["Real"] == 0:
            continue

        cor = "red" if row["Real"] == 1 else "blue"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0, y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

    fig.update_layout(
        template='plotly_white',
        hovermode='x unified',
        title=titulo
    )
    return fig

# Sem eventos falsos
fig = plot_crystallizer_unificado(
    df_crystallizer123, df_eventos_crystallizer123,
    titulo="Fe (ppm) - Crystallizers Unificados (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

# Estatísticas descritivas dos eventos

## Crystallizer #1

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer1_0a10},
    {"titulo": "IQR",              "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer1.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Definição do dataset
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]
janela_base = max(JANELAS) # Usada como referência de longo prazo (15d)

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

# 1. Extração de todas as features (Absolutas e Relativas) por evento
dados_por_classe = {0: [], 1: []}

for _, evento in df_eventos_crystallizer1.iterrows():
    ts = evento["TIMESTAMP"]
    classe = int(evento["Real"])
    feat_evento = {}
    
    # A. Calcula as absolutas para todas as janelas
    for dias in JANELAS:
        inicio = ts - pd.Timedelta(days=dias)
        mask = (df['TIMESTAMP'] >= inicio) & (df['TIMESTAMP'] < ts)
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        
        if len(v) < 2:
            for nome_stat in STATS_FUNCS:
                feat_evento[f"{nome_stat}_{dias}d"] = np.nan
        else:
            for nome_stat, func in STATS_FUNCS.items():
                feat_evento[f"{nome_stat}_{dias}d"] = float(func(v))
                
    # B. Calcula as relativas (Variação) usando a janela_base como âncora
    for dias in JANELAS:
        if dias == janela_base:
            continue
            
        media_curta = feat_evento.get(f"media_{dias}d", np.nan)
        media_longa = feat_evento.get(f"media_{janela_base}d", np.nan)
        max_curta   = feat_evento.get(f"max_{dias}d", np.nan)
        std_curta   = feat_evento.get(f"std_{dias}d", np.nan)
        std_longa   = feat_evento.get(f"std_{janela_base}d", np.nan)
        
        # Evita divisão por zero ou uso de dados faltantes
        if pd.notna(media_curta) and pd.notna(media_longa):
            feat_evento[f'aceleracao_media_{dias}d_vs_{janela_base}d'] = media_curta / (media_longa + 0.001)
            feat_evento[f'velocidade_diff_{dias}d_vs_{janela_base}d']  = media_curta - media_longa
            feat_evento[f'pico_max_{dias}d_vs_media_{janela_base}d']   = max_curta / (media_longa + 0.001)
        else:
            feat_evento[f'aceleracao_media_{dias}d_vs_{janela_base}d'] = np.nan
            feat_evento[f'velocidade_diff_{dias}d_vs_{janela_base}d']  = np.nan
            feat_evento[f'pico_max_{dias}d_vs_media_{janela_base}d']   = np.nan
            
        if pd.notna(std_curta) and pd.notna(std_longa):
            feat_evento[f'volatilidade_std_{dias}d_vs_{janela_base}d'] = std_curta / (std_longa + 0.001)
        else:
            feat_evento[f'volatilidade_std_{dias}d_vs_{janela_base}d'] = np.nan

    dados_por_classe[classe].append(feat_evento)

# Transforma em DataFrames para facilitar o corte
df_c0 = pd.DataFrame(dados_por_classe[0])
df_c1 = pd.DataFrame(dados_por_classe[1])

# 2. Teste de Mann-Whitney para cada feature
registros = []
todas_features = df_c0.columns

for feature in todas_features:
    v0 = df_c0[feature].dropna().values
    v1 = df_c1[feature].dropna().values
    
    if len(v0) < 2 or len(v1) < 2:
        continue
        
    stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
    effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
    
    registros.append({
        'feature': feature,
        'effect_size': round(effect, 4),
        'p_value': round(p, 4)
    })

df_effect = pd.DataFrame(registros).sort_values(by='effect_size', ascending=True)

# Removemos features com p-value muito alto (não significativas) opcionalmente
# df_effect = df_effect[df_effect['p_value'] < 0.05]

# Pega apenas as Top 30 features para o gráfico não ficar esmagado
df_plot = df_effect.tail(30)

# 3. Visualização (Bar Chart Horizontal)
cores = ['#2ca02c' if 'vs' in feat else '#1f77b4' for feat in df_plot['feature']]

fig = go.Figure(go.Bar(
    x=df_plot['effect_size'],
    y=df_plot['feature'],
    orientation='h',
    marker_color=cores,
    text=df_plot['effect_size'],
    textposition='outside'
))

fig.update_layout(
    title="Top 30 Features por Effect Size (Mann-Whitney) Crystallizer #1<br>"
          "<sup><span style='color:#2ca02c'>Verde = Relativas (Variação)</span> | "
          "<span style='color:#1f77b4'>Azul = Absolutas</span></sup>",
    template="plotly_white",
    xaxis_title="Effect Size (Rank-Biserial Correlation)",
    yaxis_title="Feature",
    height=800,
    margin=dict(l=250) # Espaço extra para ler os nomes longos das features
)

fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 (Eventos após 01/04/2020)"
)
fig.show()

## Crystallizer #2

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer2_0a10},
    {"titulo": "IQR",              "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer2.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Definição do dataset
# df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]
janela_base = max(JANELAS) # Usada como referência de longo prazo (15d)

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

# 1. Extração de todas as features (Absolutas e Relativas) por evento
dados_por_classe = {0: [], 1: []}

for _, evento in df_eventos_crystallizer1.iterrows():
    ts = evento["TIMESTAMP"]
    classe = int(evento["Real"])
    feat_evento = {}
    
    # A. Calcula as absolutas para todas as janelas
    for dias in JANELAS:
        inicio = ts - pd.Timedelta(days=dias)
        mask = (df['TIMESTAMP'] >= inicio) & (df['TIMESTAMP'] < ts)
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        
        if len(v) < 2:
            for nome_stat in STATS_FUNCS:
                feat_evento[f"{nome_stat}_{dias}d"] = np.nan
        else:
            for nome_stat, func in STATS_FUNCS.items():
                feat_evento[f"{nome_stat}_{dias}d"] = float(func(v))
                
    # B. Calcula as relativas (Variação) usando a janela_base como âncora
    for dias in JANELAS:
        if dias == janela_base:
            continue
            
        media_curta = feat_evento.get(f"media_{dias}d", np.nan)
        media_longa = feat_evento.get(f"media_{janela_base}d", np.nan)
        max_curta   = feat_evento.get(f"max_{dias}d", np.nan)
        std_curta   = feat_evento.get(f"std_{dias}d", np.nan)
        std_longa   = feat_evento.get(f"std_{janela_base}d", np.nan)
        
        # Evita divisão por zero ou uso de dados faltantes
        if pd.notna(media_curta) and pd.notna(media_longa):
            feat_evento[f'aceleracao_media_{dias}d_vs_{janela_base}d'] = media_curta / (media_longa + 0.001)
            feat_evento[f'velocidade_diff_{dias}d_vs_{janela_base}d']  = media_curta - media_longa
            feat_evento[f'pico_max_{dias}d_vs_media_{janela_base}d']   = max_curta / (media_longa + 0.001)
        else:
            feat_evento[f'aceleracao_media_{dias}d_vs_{janela_base}d'] = np.nan
            feat_evento[f'velocidade_diff_{dias}d_vs_{janela_base}d']  = np.nan
            feat_evento[f'pico_max_{dias}d_vs_media_{janela_base}d']   = np.nan
            
        if pd.notna(std_curta) and pd.notna(std_longa):
            feat_evento[f'volatilidade_std_{dias}d_vs_{janela_base}d'] = std_curta / (std_longa + 0.001)
        else:
            feat_evento[f'volatilidade_std_{dias}d_vs_{janela_base}d'] = np.nan

    dados_por_classe[classe].append(feat_evento)

# Transforma em DataFrames para facilitar o corte
df_c0 = pd.DataFrame(dados_por_classe[0])
df_c1 = pd.DataFrame(dados_por_classe[1])

# 2. Teste de Mann-Whitney para cada feature
registros = []
todas_features = df_c0.columns

for feature in todas_features:
    v0 = df_c0[feature].dropna().values
    v1 = df_c1[feature].dropna().values
    
    if len(v0) < 2 or len(v1) < 2:
        continue
        
    stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
    effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
    
    registros.append({
        'feature': feature,
        'effect_size': round(effect, 4),
        'p_value': round(p, 4)
    })

df_effect = pd.DataFrame(registros).sort_values(by='effect_size', ascending=True)

# Removemos features com p-value muito alto (não significativas) opcionalmente
# df_effect = df_effect[df_effect['p_value'] < 0.05]

# Pega apenas as Top 30 features para o gráfico não ficar esmagado
df_plot = df_effect.tail(30)

# 3. Visualização (Bar Chart Horizontal)
cores = ['#2ca02c' if 'vs' in feat else '#1f77b4' for feat in df_plot['feature']]

fig = go.Figure(go.Bar(
    x=df_plot['effect_size'],
    y=df_plot['feature'],
    orientation='h',
    marker_color=cores,
    text=df_plot['effect_size'],
    textposition='outside'
))

fig.update_layout(
    title="Top 30 Features por Effect Size (Mann-Whitney) Crystallizer #2<br>"
          "<sup><span style='color:#2ca02c'>Verde = Relativas (Variação)</span> | "
          "<span style='color:#1f77b4'>Azul = Absolutas</span></sup>",
    template="plotly_white",
    xaxis_title="Effect Size (Rank-Biserial Correlation)",
    yaxis_title="Feature",
    height=800,
    margin=dict(l=250) # Espaço extra para ler os nomes longos das features
)

fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 (Eventos após 01/04/2020)"
)
fig.show()

## Crystallizer #3

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer3_0a10},
    {"titulo": "IQR",              "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer3.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Definição do dataset
# df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]
janela_base = max(JANELAS) # Usada como referência de longo prazo (15d)

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

# 1. Extração de todas as features (Absolutas e Relativas) por evento
dados_por_classe = {0: [], 1: []}

for _, evento in df_eventos_crystallizer1.iterrows():
    ts = evento["TIMESTAMP"]
    classe = int(evento["Real"])
    feat_evento = {}
    
    # A. Calcula as absolutas para todas as janelas
    for dias in JANELAS:
        inicio = ts - pd.Timedelta(days=dias)
        mask = (df['TIMESTAMP'] >= inicio) & (df['TIMESTAMP'] < ts)
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        
        if len(v) < 2:
            for nome_stat in STATS_FUNCS:
                feat_evento[f"{nome_stat}_{dias}d"] = np.nan
        else:
            for nome_stat, func in STATS_FUNCS.items():
                feat_evento[f"{nome_stat}_{dias}d"] = float(func(v))
                
    # B. Calcula as relativas (Variação) usando a janela_base como âncora
    for dias in JANELAS:
        if dias == janela_base:
            continue
            
        media_curta = feat_evento.get(f"media_{dias}d", np.nan)
        media_longa = feat_evento.get(f"media_{janela_base}d", np.nan)
        max_curta   = feat_evento.get(f"max_{dias}d", np.nan)
        std_curta   = feat_evento.get(f"std_{dias}d", np.nan)
        std_longa   = feat_evento.get(f"std_{janela_base}d", np.nan)
        
        # Evita divisão por zero ou uso de dados faltantes
        if pd.notna(media_curta) and pd.notna(media_longa):
            feat_evento[f'aceleracao_media_{dias}d_vs_{janela_base}d'] = media_curta / (media_longa + 0.001)
            feat_evento[f'velocidade_diff_{dias}d_vs_{janela_base}d']  = media_curta - media_longa
            feat_evento[f'pico_max_{dias}d_vs_media_{janela_base}d']   = max_curta / (media_longa + 0.001)
        else:
            feat_evento[f'aceleracao_media_{dias}d_vs_{janela_base}d'] = np.nan
            feat_evento[f'velocidade_diff_{dias}d_vs_{janela_base}d']  = np.nan
            feat_evento[f'pico_max_{dias}d_vs_media_{janela_base}d']   = np.nan
            
        if pd.notna(std_curta) and pd.notna(std_longa):
            feat_evento[f'volatilidade_std_{dias}d_vs_{janela_base}d'] = std_curta / (std_longa + 0.001)
        else:
            feat_evento[f'volatilidade_std_{dias}d_vs_{janela_base}d'] = np.nan

    dados_por_classe[classe].append(feat_evento)

# Transforma em DataFrames para facilitar o corte
df_c0 = pd.DataFrame(dados_por_classe[0])
df_c1 = pd.DataFrame(dados_por_classe[1])

# 2. Teste de Mann-Whitney para cada feature
registros = []
todas_features = df_c0.columns

for feature in todas_features:
    v0 = df_c0[feature].dropna().values
    v1 = df_c1[feature].dropna().values
    
    if len(v0) < 2 or len(v1) < 2:
        continue
        
    stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
    effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
    
    registros.append({
        'feature': feature,
        'effect_size': round(effect, 4),
        'p_value': round(p, 4)
    })

df_effect = pd.DataFrame(registros).sort_values(by='effect_size', ascending=True)

# Removemos features com p-value muito alto (não significativas) opcionalmente
# df_effect = df_effect[df_effect['p_value'] < 0.05]

# Pega apenas as Top 30 features para o gráfico não ficar esmagado
df_plot = df_effect.tail(30)

# 3. Visualização (Bar Chart Horizontal)
cores = ['#2ca02c' if 'vs' in feat else '#1f77b4' for feat in df_plot['feature']]

fig = go.Figure(go.Bar(
    x=df_plot['effect_size'],
    y=df_plot['feature'],
    orientation='h',
    marker_color=cores,
    text=df_plot['effect_size'],
    textposition='outside'
))

fig.update_layout(
    title="Top 30 Features por Effect Size (Mann-Whitney) Crystallizer #3<br>"
          "<sup><span style='color:#2ca02c'>Verde = Relativas (Variação)</span> | "
          "<span style='color:#1f77b4'>Azul = Absolutas</span></sup>",
    template="plotly_white",
    xaxis_title="Effect Size (Rank-Biserial Correlation)",
    yaxis_title="Feature",
    height=800,
    margin=dict(l=250) # Espaço extra para ler os nomes longos das features
)

fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 (Eventos após 01/04/2020)"
)
fig.show()

# Comparação dos 3 reatores

## Verificando effect size por reator

In [ ]:
crystallizers_config = [
    # C1
    (df_crystallizer1,        df_eventos_crystallizer1, "C1 Original"),
    (df_crystallizer1_0a10,   df_eventos_crystallizer1, "C1 0-10"),
    (df_crystallizer1_iqr,    df_eventos_crystallizer1, "C1 IQR"),
    (df_crystallizer1_hampel, df_eventos_crystallizer1, "C1 Hampel"),
    # C2
    (df_crystallizer2,        df_eventos_crystallizer2, "C2 Original"),
    (df_crystallizer2_0a10,   df_eventos_crystallizer2, "C2 0-10"),
    (df_crystallizer2_iqr,    df_eventos_crystallizer2, "C2 IQR"),
    (df_crystallizer2_hampel, df_eventos_crystallizer2, "C2 Hampel"),
    # C3
    (df_crystallizer3,        df_eventos_crystallizer3, "C3 Original"),
    (df_crystallizer3_0a10,   df_eventos_crystallizer3, "C3 0-10"),
    (df_crystallizer3_iqr,    df_eventos_crystallizer3, "C3 IQR"),
    (df_crystallizer3_hampel, df_eventos_crystallizer3, "C3 Hampel"),
    # Unificado
    (df_crystallizer123,        df_eventos_crystallizer123, "Unificado Original"),
    (df_crystallizer123_0a10,   df_eventos_crystallizer123, "Unificado 0-10"),
    (df_crystallizer123_iqr,    df_eventos_crystallizer123, "Unificado IQR"),
    (df_crystallizer123_hampel, df_eventos_crystallizer123, "Unificado Hampel"),
]

STATS_FUNCS = {
    'media'   : np.mean,
    'mediana' : np.median,
    'std'     : np.std,
    'max'     : np.max,
    'p75'     : lambda x: np.percentile(x, 75),
    'p90'     : lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range'   : lambda x: np.max(x) - np.min(x),
}

registros_todos = []
for df_c, df_ev, nome_c in crystallizers_config:
    for DIAS_JANELA in JANELAS:
        stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}
        for _, evento in df_ev.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_c['TIMESTAMP'] >= inicio) & (df_c['TIMESTAMP'] < ts)
            v      = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) < 2:
                continue
            classe = int(evento["Real"])
            for nome_stat, func in STATS_FUNCS.items():
                stat_vals[nome_stat][classe].append(func(v))

        for nome_stat in STATS_FUNCS:
            v0 = np.array(stat_vals[nome_stat][0])
            v1 = np.array(stat_vals[nome_stat][1])
            if len(v0) < 2 or len(v1) < 2:
                continue
            stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
            effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
            registros_todos.append({
                'crystallizer': nome_c,
                'janela'      : f"{DIAS_JANELA}d",
                'feature'     : nome_stat,
                'effect_size' : round(effect, 4),
                'p_value'     : round(p, 4),
            })

df_effect_todos = pd.DataFrame(registros_todos)

# Heatmap — 4 origens (C1, C2, C3, Unificado) × 4 métodos = 16 subplots
n_cols = 4  # Original, 0-10, IQR, Hampel
n_rows = 4  # C1, C2, C3, Unificado
nomes  = [c[2] for c in crystallizers_config]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=nomes,
    vertical_spacing=0.06
)

for idx, (_, _, nome_c) in enumerate(crystallizers_config):
    row_idx = idx // n_cols + 1
    col_idx = idx % n_cols + 1

    subset = df_effect_todos[df_effect_todos['crystallizer'] == nome_c]
    pivot  = subset.pivot(index='feature', columns='janela', values='effect_size')
    pivot  = pivot[[f"{d}d" for d in JANELAS]]

    fig.add_trace(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=np.round(pivot.values, 3),
        texttemplate="%{text}",
        showscale=(col_idx == n_cols and row_idx == n_rows)
    ), row=row_idx, col=col_idx)

fig.update_layout(
    title="Effect Size comparativo — C1, C2, C3 e Unificado × Original, 0-10, IQR, Hampel",
    template="plotly_white",
    height=400 * n_rows
)
fig.show()

## Correlação cruzada entre os crystallizers

In [ ]:
def add_scatter_regressao(fig, x, y, row, col):
    lr = LinearRegression().fit(x.reshape(-1, 1), y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))
    r2 = lr.score(x.reshape(-1, 1), y)
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(size=6, opacity=0.6, color='steelblue'),
        showlegend=False
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode='lines',
        line=dict(color='red', width=2),
        showlegend=False
    ), row=row, col=col)
    fig.add_annotation(
        x=x.min(), y=y.max(),
        text=f"R² = {r2:.3f}",
        showarrow=False,
        font=dict(size=12, color='red'),
        xanchor='left',
        row=row, col=col
    )

freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

    print(f"\n{'='*55}")
    print(f"Método: {m['titulo']}")
    print(m["df_corr"].corr(method="pearson").round(3).to_string())


n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=3,
    subplot_titles=[
        f"{a} vs {b} — {m['titulo']}"
        for m in metodos
        for a, b in pares
    ],
    vertical_spacing=0.06
)

for row_idx, m in enumerate(metodos, start=1):
    for col_idx, (a, b) in enumerate(pares, start=1):
        add_scatter_regressao(fig, m["df_corr"][a].values, m["df_corr"][b].values, row=row_idx, col=col_idx)
        fig.update_xaxes(title_text=a, row=row_idx, col=col_idx)
        fig.update_yaxes(title_text=b, row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template='plotly_white',
    title="Correlação cruzada — C1, C2, C3 × Original, 0-10, IQR, Hampel"
)
fig.show()

## Correlação cruzada com lags
Verificando se um reator tem influência sobre outro

In [ ]:
MAX_LAG = 15
lags = range(-MAX_LAG, MAX_LAG + 1)
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

# Prepara df_corr para cada método
for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

# Cross-correlação com lag
n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Cross-correlação com Defasagem — {m['titulo']}" for m in metodos],
    vertical_spacing=0.08
)

for row_idx, m in enumerate(metodos, start=1):
    df_c = m["df_corr"]
    resultados_lag = []

    for a, b in pares:
        x = df_c[a].values
        y = df_c[b].values
        for lag in lags:
            if lag < 0:
                xs, ys = x[:lag],  y[-lag:]
            elif lag > 0:
                xs, ys = x[lag:],  y[:-lag]
            else:
                xs, ys = x, y
            r, p = pearsonr(xs, ys)
            resultados_lag.append({
                "par": f"{a} vs {b}", "lag_dias": lag * 3,
                "r": round(r, 4), "p": round(p, 6)
            })

    df_lag = pd.DataFrame(resultados_lag)

    for par in df_lag["par"].unique():
        sub = df_lag[df_lag["par"] == par]
        fig.add_trace(go.Scatter(
            x=sub["lag_dias"], y=sub["r"],
            mode="lines", name=par,
            line=dict(width=2),
            legendgroup=par,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.add_hline(y=0, line_color="gray", line_width=0.5, row=row_idx, col=1)
    fig.update_yaxes(title_text="Pearson r", row=row_idx, col=1)
    fig.update_xaxes(title_text="Defasagem (dias)", row=row_idx, col=1)

    # Lag de máxima correlação
    print(f"\nMétodo: {m['titulo']} — Lag de máxima correlação:")
    for par in df_lag["par"].unique():
        sub  = df_lag[df_lag["par"] == par]
        best = sub.loc[sub["r"].idxmax()]
        print(f"  {par}: lag={best['lag_dias']:.0f} dias  |  r={best['r']:.4f}")

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    hovermode="x unified",
    title="Correlação cruzada com Defasagem entre Reatores<br>"
          "<sup>Pico em lag≠0 indica que um reator influencia o outro, negativo: A influencia B  |  positivo: B influencia A</sup>"
)
fig.show()

# Clusterização (Não supervisionada)

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

STATS_FUNCS_CLUSTERING = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max,
    'p75':     lambda x: np.percentile(x, 75),
    'p90':     lambda x: np.percentile(x, 90),
    'range':   lambda x: np.max(x) - np.min(x),
}

configs_crystallizer = {
    "C1": {
        "Original":      (df_crystallizer1,        df_eventos_crystallizer1),
        "Intervalo 0-10":(df_crystallizer1_0a10,   df_eventos_crystallizer1),
        "IQR":           (df_crystallizer1_iqr,    df_eventos_crystallizer1),
        "Hampel":        (df_crystallizer1_hampel, df_eventos_crystallizer1),
    },
    "C2": {
        "Original":      (df_crystallizer2,        df_eventos_crystallizer2),
        "Intervalo 0-10":(df_crystallizer2_0a10,   df_eventos_crystallizer2),
        "IQR":           (df_crystallizer2_iqr,    df_eventos_crystallizer2),
        "Hampel":        (df_crystallizer2_hampel, df_eventos_crystallizer2),
    },
    "C3": {
        "Original":      (df_crystallizer3,        df_eventos_crystallizer3),
        "Intervalo 0-10":(df_crystallizer3_0a10,   df_eventos_crystallizer3),
        "IQR":           (df_crystallizer3_iqr,    df_eventos_crystallizer3),
        "Hampel":        (df_crystallizer3_hampel, df_eventos_crystallizer3),
    },
    "C123": {
        "Original":      (df_crystallizer123,        df_eventos_crystallizer123),
        "Intervalo 0-10":(df_crystallizer123_0a10,   df_eventos_crystallizer123),
        "IQR":           (df_crystallizer123_iqr,    df_eventos_crystallizer123),
        "Hampel":        (df_crystallizer123_hampel, df_eventos_crystallizer123),
    },
}

def extrair_features(nome_c, df_medicoes, df_eventos, janelas, stats_funcs):
    dataset_linhas = []
    for _, evento in df_eventos.iterrows():
        ts     = evento["TIMESTAMP"]
        classe = int(evento["Real"])
        features = {
            'Crystallizer':     nome_c,
            'TIMESTAMP_Evento': ts,
            'Real':             classe,
        }
        for dias in janelas:
            inicio = ts - pd.Timedelta(days=dias)
            mask   = (df_medicoes['TIMESTAMP'] >= inicio) & \
                     (df_medicoes['TIMESTAMP'] <  ts)
            y_ppm  = df_medicoes[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(y_ppm) < 2:
                for nome_stat in stats_funcs:
                    features[f"{nome_stat}_{dias}d"] = np.nan
                continue
            for nome_stat, func in stats_funcs.items():
                features[f"{nome_stat}_{dias}d"] = float(func(y_ppm))

        if not any(not np.isnan(features.get(f"media_{d}d", np.nan))
                   for d in janelas):
            continue
        dataset_linhas.append(features)

    colunas_meta = ['Crystallizer', 'TIMESTAMP_Evento', 'Real']
    df = pd.DataFrame(dataset_linhas)
    colunas_features = sorted(
        [c for c in df.columns if c not in colunas_meta],
        key=lambda c: (c.split('_')[0], int(c.split('_')[-1].replace('d', '')))
    )
    return df[colunas_meta + colunas_features], colunas_features


def pipeline_clustering(df_feat, colunas_features, label_dataset):
    # Imputação
    imputer  = SimpleImputer(strategy='median')
    X_imp    = pd.DataFrame(
        imputer.fit_transform(df_feat[colunas_features]),
        columns=colunas_features, index=df_feat.index
    )

    # Scaling
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    # Grid search
    resultados = []
    for n_clusters in range(2, 8):
        modelos = {
            'Hierarchical (ward)':     AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
            'Hierarchical (complete)': AgglomerativeClustering(n_clusters=n_clusters, linkage='complete'),
            'Hierarchical (average)':  AgglomerativeClustering(n_clusters=n_clusters, linkage='average'),
            'GMM':                     GaussianMixture(n_components=n_clusters, random_state=42, n_init=10),
            'Bisecting KMeans':        BisectingKMeans(n_clusters=n_clusters, random_state=42, n_init=10),
            'Spectral':                SpectralClustering(n_clusters=n_clusters,
                                                          affinity='nearest_neighbors',
                                                          n_neighbors=10, random_state=42),
        }
        for nome, modelo in modelos.items():
            labels = modelo.fit_predict(X_scaled)
            ari    = adjusted_rand_score(df_feat['Real'], labels)
            nmi    = normalized_mutual_info_score(df_feat['Real'], labels)
            resultados.append({
                'Dataset': label_dataset, 'Modelo': nome,
                'n_clusters': n_clusters,
                'ARI': round(ari, 4), 'NMI': round(nmi, 4),
            })

    df_grid  = pd.DataFrame(resultados).sort_values('ARI', ascending=False)
    melhor   = df_grid.iloc[0]
    melhor_n = int(melhor['n_clusters'])
    melhor_mod = melhor['Modelo']

    # Clusterização final
    modelos_final = {
        'Hierarchical (ward)':     AgglomerativeClustering(n_clusters=melhor_n, linkage='ward'),
        'Hierarchical (complete)': AgglomerativeClustering(n_clusters=melhor_n, linkage='complete'),
        'Hierarchical (average)':  AgglomerativeClustering(n_clusters=melhor_n, linkage='average'),
        'GMM':                     GaussianMixture(n_components=melhor_n, random_state=42, n_init=10),
        'Bisecting KMeans':        BisectingKMeans(n_clusters=melhor_n, random_state=42, n_init=10),
        'Spectral':                SpectralClustering(n_clusters=melhor_n,
                                                      affinity='nearest_neighbors',
                                                      n_neighbors=10, random_state=42),
    }
    df_feat = df_feat.copy()
    df_feat['Cluster'] = modelos_final[melhor_mod].fit_predict(X_scaled)

    ari_f = adjusted_rand_score(df_feat['Real'], df_feat['Cluster'])
    nmi_f = normalized_mutual_info_score(df_feat['Real'], df_feat['Cluster'])

    return {
        'df_features': df_feat,
        'X_scaled':    X_scaled,
        'df_grid':     df_grid,
        'melhor_mod':  melhor_mod,
        'melhor_n':    melhor_n,
        'ari':         ari_f,
        'nmi':         nmi_f,
    }

## Crystallizer #1

In [ ]:
CRYSTALLIZER = "C1"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

## Crystallizer #2

In [ ]:
CRYSTALLIZER = "C2"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

## Crystallizer #3

In [ ]:
CRYSTALLIZER = "C3"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

## Crystallizer #1 #2 #3

In [ ]:
CRYSTALLIZER = "C123"  # "C1" | "C2" | "C3"

resultados_por_dataset = {}

for nome_dataset, (df_med, df_ev) in configs_crystallizer[CRYSTALLIZER].items():

    df_feat, cols_feat = extrair_features(
        CRYSTALLIZER, df_med, df_ev, JANELAS, STATS_FUNCS_CLUSTERING
    )
    res = pipeline_clustering(df_feat, cols_feat, nome_dataset)
    resultados_por_dataset[nome_dataset] = res

    print(f"\n{'='*55}")
    print(f"{CRYSTALLIZER} — {nome_dataset}")
    print(f"  Melhor modelo : {res['melhor_mod']}  (k={res['melhor_n']})")
    print(f"  ARI           : {res['ari']:.4f}")
    print(f"  NMI           : {res['nmi']:.4f}")
    print("\n  Tabela de contingência:")
    print(pd.crosstab(
        res['df_features']['Cluster'], res['df_features']['Real'],
        rownames=['Cluster'], colnames=['Real'],
        margins=True, margins_name='Total'
    ))

In [ ]:
n_datasets = len(resultados_por_dataset)
nomes_datasets = list(resultados_por_dataset.keys())

fig = make_subplots(
    rows=n_datasets, cols=2,
    subplot_titles=[
        titulo
        for nd in nomes_datasets
        for titulo in [
            f"{nd} — Clusters ({resultados_por_dataset[nd]['melhor_mod']}, "
            f"k={resultados_por_dataset[nd]['melhor_n']})",
            f"{nd} — Label Real"
        ]
    ],
    vertical_spacing=0.06
)

cores_cluster = px.colors.qualitative.Plotly
cores_real    = {'0': '#4878CF', '1': '#D65F5F'}
nomes_real    = {'0': 'Falso Positivo', '1': 'Contaminação Real'}
simbolos      = {'0': 'circle', '1': 'diamond'}

for row_idx, nome_dataset in enumerate(nomes_datasets, start=1):
    res      = resultados_por_dataset[nome_dataset]
    df_feat  = res['df_features']
    X_scaled = res['X_scaled']

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100

    df_plot = pd.DataFrame({
        'PC1':     X_pca[:, 0],
        'PC2':     X_pca[:, 1],
        'Cluster': df_feat['Cluster'].astype(str),
        'Real':    df_feat['Real'].astype(str),
    })

    # Subplot esquerdo — clusters
    for cluster_id in sorted(df_plot['Cluster'].unique()):
        mask = df_plot['Cluster'] == cluster_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=f'Cluster {cluster_id}',
            marker=dict(
                size=8,
                color=cores_cluster[int(cluster_id) % len(cores_cluster)],
                opacity=0.8, line=dict(width=0.5, color='white')
            ),
            legendgroup=f'cluster_{cluster_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Subplot direito — label real
    for real_id in ['0', '1']:
        mask = df_plot['Real'] == real_id
        fig.add_trace(go.Scatter(
            x=df_plot.loc[mask, 'PC1'], y=df_plot.loc[mask, 'PC2'],
            mode='markers',
            name=nomes_real[real_id],
            marker=dict(
                size=8, color=cores_real[real_id],
                symbol=simbolos[real_id], opacity=0.85,
                line=dict(width=0.5, color='white')
            ),
            legendgroup=f'real_{real_id}',
            showlegend=(row_idx == 1)
        ), row=row_idx, col=2)

    for col in [1, 2]:
        fig.update_xaxes(title_text=f'PC1 ({var_pc1:.1f}%)', row=row_idx, col=col)
        fig.update_yaxes(title_text=f'PC2 ({var_pc2:.1f}%)', row=row_idx, col=col)

fig.update_layout(
    title=f'PCA 2D — {CRYSTALLIZER}',
    template='plotly_white',
    height=500 * n_datasets,
    legend=dict(groupclick='toggleitem')
)
fig.show()

# Classificação (Abordagem Supervisionada)

In [ ]:
JANELAS       = [15, 12, 9, 6, 3]
N_SPLITS      = 5
RANDOM_STATE  = 42
TEST_SIZE     = 0.2

STATS_FUNCS = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max,
    'p75':     lambda x: np.percentile(x, 75),
    'p90':     lambda x: np.percentile(x, 90),
    'range':   lambda x: np.max(x) - np.min(x),
}

MAP_MEDICOES = {
    "Original":       {"C1": df_crystallizer1,        "C2": df_crystallizer2,        "C3": df_crystallizer3},
    "Intervalo 0-10": {"C1": df_crystallizer1_0a10,   "C2": df_crystallizer2_0a10,   "C3": df_crystallizer3_0a10},
    "IQR":            {"C1": df_crystallizer1_iqr,    "C2": df_crystallizer2_iqr,    "C3": df_crystallizer3_iqr},
    "Hampel":         {"C1": df_crystallizer1_hampel, "C2": df_crystallizer2_hampel, "C3": df_crystallizer3_hampel},
}

CONFIGS = {
    "C1": {
        "Original":       (df_crystallizer1,        df_eventos_crystallizer1),
        "Intervalo 0-10": (df_crystallizer1_0a10,   df_eventos_crystallizer1),
        "IQR":            (df_crystallizer1_iqr,    df_eventos_crystallizer1),
        "Hampel":         (df_crystallizer1_hampel, df_eventos_crystallizer1),
    },
    "C2": {
        "Original":       (df_crystallizer2,        df_eventos_crystallizer2),
        "Intervalo 0-10": (df_crystallizer2_0a10,   df_eventos_crystallizer2),
        "IQR":            (df_crystallizer2_iqr,    df_eventos_crystallizer2),
        "Hampel":         (df_crystallizer2_hampel, df_eventos_crystallizer2),
    },
    "C3": {
        "Original":       (df_crystallizer3,        df_eventos_crystallizer3),
        "Intervalo 0-10": (df_crystallizer3_0a10,   df_eventos_crystallizer3),
        "IQR":            (df_crystallizer3_iqr,    df_eventos_crystallizer3),
        "Hampel":         (df_crystallizer3_hampel, df_eventos_crystallizer3),
    },
}

def extrair_features(crystallizer, df_medicoes, df_eventos, janelas, stats_funcs, map_medicoes_unif=None):
    dataset_linhas = []
    janela_base = max(janelas)
    colunas_meta = ['Crystallizer', 'TIMESTAMP_Evento', 'Real']
    
    for _, evento in df_eventos.iterrows():
        ts     = evento["TIMESTAMP"]
        classe = int(evento["Real"])
        cryst  = evento.get("Crystallizer", crystallizer)
        df_med = map_medicoes_unif[cryst] if map_medicoes_unif else df_medicoes

        features = {'Crystallizer': cryst, 'TIMESTAMP_Evento': ts, 'Real': classe}
        
        for dias in janelas:
            inicio = ts - pd.Timedelta(days=dias)
            mask   = (df_med['TIMESTAMP'] >= inicio) & (df_med['TIMESTAMP'] < ts)
            y_ppm  = df_med[mask]["Resultado de Ferro (ppm)"].dropna().values
            
            if len(y_ppm) < 2:
                for nome_stat in stats_funcs:
                    features[f"{nome_stat}_{dias}d"] = np.nan
                continue
                
            for nome_stat, func in stats_funcs.items():
                features[f"{nome_stat}_{dias}d"] = float(func(y_ppm))

        if pd.isna(features.get(f"media_{janela_base}d", np.nan)):
            continue
            
        for dias in janelas:
            if dias == janela_base:
                continue
                
            features[f'acel_media_{dias}d_vs_{janela_base}d'] = features[f'media_{dias}d'] / (features[f'media_{janela_base}d'] + 0.001)
            features[f'vel_diff_{dias}d_vs_{janela_base}d'] = features[f'media_{dias}d'] - features[f'media_{janela_base}d']
            features[f'pico_max_{dias}d_vs_media_{janela_base}d'] = features[f'max_{dias}d'] / (features[f'media_{janela_base}d'] + 0.001)
            features[f'volatilidade_std_{dias}d_vs_{janela_base}d'] = features[f'std_{dias}d'] / (features[f'std_{janela_base}d'] + 0.001)

        janela_curta = min(janelas)
        chaves_para_remover = [k for k in features.keys() if ('d' in k and 'vs' not in k and f'_{janela_curta}d' not in k)]
        for k in chaves_para_remover:
            del features[k]

        dataset_linhas.append(features)

    if not dataset_linhas:
        return pd.DataFrame(columns=colunas_meta), []

    df = pd.DataFrame(dataset_linhas)
    colunas_features = [c for c in df.columns if c not in colunas_meta]
    return df[colunas_meta + colunas_features], colunas_features


def dividir_dados(df_features, colunas_features, test_size=TEST_SIZE, incluir_crystallizer=False):
    X = df_features.sort_values('TIMESTAMP_Evento').copy()
    
    if incluir_crystallizer:
        le = LabelEncoder()
        X['Crystallizer_enc'] = le.fit_transform(X['Crystallizer'])
        
    y = X['Real']
    eventos_pos = X[X['Real'] == 1]
    
    if len(eventos_pos) >= 2:
        idx_corte_pos = int(len(eventos_pos) * (1 - test_size))
        ts_corte = eventos_pos.iloc[idx_corte_pos]['TIMESTAMP_Evento']
        train_mask = X['TIMESTAMP_Evento'] < ts_corte
        test_mask  = X['TIMESTAMP_Evento'] >= ts_corte
    else:
        split_idx = int(len(X) * (1 - test_size))
        train_mask = np.arange(len(X)) < split_idx
        test_mask  = np.arange(len(X)) >= split_idx

    X_train = X[train_mask][colunas_features]
    X_test  = X[test_mask][colunas_features]
    y_train = y[train_mask]
    y_test  = y[test_mask]
    
    return X_train, X_test, y_train, y_test, X[colunas_features], y


def selecionar_features(X_train, y_train, colunas_brutas, k_features=8, limite_correlacao=0.85):
    X = X_train[colunas_brutas].copy()
    for col in X.columns:
        X[col] = X[col].fillna(X[col].median())
        
    matriz_corr = X.corr(method='spearman').abs()
    upper = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))
    colunas_para_dropar = [column for column in upper.columns if any(upper[column] > limite_correlacao)]
    X_sem_corr = X.drop(columns=colunas_para_dropar)
    
    n_features_disponiveis = X_sem_corr.shape[1]
    k_final = min(k_features, n_features_disponiveis)
    
    if k_final == 0: return []
        
    seletor = SelectKBest(score_func=f_classif, k=k_final)
    seletor.fit(X_sem_corr, y_train)
    features_selecionadas = X_sem_corr.columns[seletor.get_support()].tolist()
    return features_selecionadas


def definir_modelos_e_grids(y_train):
    ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    return {
        'Logistic Regression': (
            LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
            {'clf__C': [0.01, 0.1, 1.0]}
        ),
        'Random Forest': (
            RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
            {'clf__n_estimators': [100, 300], 'clf__max_depth': [3, 5, 7]}
        ),
        'XGBoost': (
            XGBClassifier(scale_pos_weight=ratio, eval_metric='aucpr', random_state=RANDOM_STATE, verbosity=0),
            {'clf__n_estimators': [100, 200], 'clf__max_depth': [3, 5], 'clf__learning_rate': [0.01, 0.05]}
        ),
        'SVM RBF': (
            SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
            {'clf__C': [0.1, 1.0, 10.0]}
        ),
    }


def avaliar_cv_com_grid(X_train, y_train, modelos_grids, usar_smote, n_splits=N_SPLITS):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=False)
    scoring = {
        'f1':        make_scorer(f1_score, zero_division=0),
        'pr_auc':    'average_precision',
        'roc_auc':   'roc_auc',
        'recall':    make_scorer(recall_score, zero_division=0),
        'precision': make_scorer(precision_score, zero_division=0),
    }
    resultados, melhores_estimadores = [], {}
    n_falhas_treino = sum(y_train == 1)
    k_vizinhos = min(3, n_falhas_treino - 1) if n_falhas_treino > 1 else 1
    
    for nome, (modelo, grid) in modelos_grids.items():
        if usar_smote:
            pipe = ImbPipeline([
                ('imputer', SimpleImputer(strategy='median')), 
                ('scaler', StandardScaler()), 
                ('smote', SMOTE(k_neighbors=k_vizinhos, random_state=RANDOM_STATE)),
                ('clf', modelo)
            ])
        else:
            pipe = SklearnPipeline([
                ('imputer', SimpleImputer(strategy='median')), 
                ('scaler', StandardScaler()), 
                ('clf', modelo)
            ])
            
        gs = GridSearchCV(pipe, param_grid=grid, cv=cv, scoring=scoring, refit='pr_auc', n_jobs=-1)
        gs.fit(X_train, y_train)
        
        melhores_estimadores[nome] = gs.best_estimator_
        idx_best = gs.best_index_
        res_cv = gs.cv_results_
        
        resultados.append({
            'Modelo':    nome,
            'F1':        round(res_cv['mean_test_f1'][idx_best], 4),
            'PR_AUC':    round(res_cv['mean_test_pr_auc'][idx_best], 4),
            'ROC_AUC':   round(res_cv['mean_test_roc_auc'][idx_best], 4),
            'Recall':    round(res_cv['mean_test_recall'][idx_best], 4),
            'Precision': round(res_cv['mean_test_precision'][idx_best], 4),
            'Melhor_Params': str(gs.best_params_)
        })
        
    return pd.DataFrame(resultados).sort_values('PR_AUC', ascending=False), melhores_estimadores


def avaliar_teste(melhor_pipe, X_test, y_test, best_thr=0.4):
    probs_test = melhor_pipe.predict_proba(X_test)[:, 1]
    y_pred_test = (probs_test >= best_thr).astype(int)
    return melhor_pipe, best_thr, probs_test, y_pred_test


def plotar_resultados(resultados_por_tratamento, modo, df_cv_consolidado):
    tratamentos = list(resultados_por_tratamento.keys())
    n = len(tratamentos)

    fig_cm = make_subplots(
        rows=1, cols=n,
        subplot_titles=[f"{modo} — {t}" for t in tratamentos]
    )

    for col_idx, tratamento in enumerate(tratamentos, start=1):
        r = resultados_por_tratamento[tratamento]
        cm = confusion_matrix(r["y_test"], r["y_pred_test"])

        labels = ["Falso Positivo", "Contaminação"]
        fig_cm.add_trace(go.Heatmap(
            z=cm, x=labels, y=labels, colorscale="Blues",
            showscale=(col_idx == n), text=cm, texttemplate="%{text}", textfont=dict(size=14)
        ), row=1, col=col_idx)

        fig_cm.update_xaxes(title_text="Predito",  row=1, col=col_idx)
        fig_cm.update_yaxes(title_text="Real",     row=1, col=col_idx)

    fig_cm.update_layout(height=400, template="plotly_white", title=f"Matriz de Confusão — {modo} (Pipeline Completo)")
    fig_cm.show()

## Crystallizer #1

In [ ]:
MODO = "C1"
TRATAMENTO_ALVO = "IQR" # Escolha o tratamento base (Original, IQR, etc.) para rodar os testes

print(f"\nIniciando testes comparativos no {MODO} usando tratamento {TRATAMENTO_ALVO}\n")

df_med, df_ev = CONFIGS[MODO][TRATAMENTO_ALVO]
df_feat, cols_feat_brutas = extrair_features(MODO, df_med, df_ev, JANELAS, STATS_FUNCS)

abordagens = [
    {"nome": "Base (Sem SMOTE, Sem FS)", "smote": False, "fs": False},
    {"nome": "Apenas SMOTE", "smote": True, "fs": False},
    {"nome": "Apenas Feature Selection", "smote": False, "fs": True},
    {"nome": "SMOTE + Feature Selection", "smote": True, "fs": True}
]

relatorio_final = []
resultados_para_plot = {} # <--- DICIONÁRIO CRIADO AQUI

if df_feat.empty or len(df_feat) < N_SPLITS + 2:
    print("Dados insuficientes.")
else:
    for abordagem in abordagens:
        nome_ab = abordagem['nome']
        print(f"\n{'='*55}")
        print(f"Executando: {nome_ab}")
        print(f"{'='*55}")

        X_train_b, X_test_b, y_train, y_test, _, _ = dividir_dados(df_feat, cols_feat_brutas, test_size=TEST_SIZE)
        
        # Lógica da Feature Selection
        if abordagem['fs']:
            cols_ativas = selecionar_features(X_train_b, y_train, cols_feat_brutas, k_features=10)
            if not cols_ativas: 
                print(f"  -> AVISO: Nenhuma feature sobreviveu à seleção em {nome_ab}. Pulando.")
                continue
        else:
            cols_ativas = cols_feat_brutas
            
        X_train, X_test = X_train_b[cols_ativas], X_test_b[cols_ativas]

        modelos_grids = definir_modelos_e_grids(y_train)
        df_cv, melhores_estimadores = avaliar_cv_com_grid(X_train, y_train, modelos_grids, usar_smote=abordagem['smote'])
        
        melhor_nome = df_cv.iloc[0]['Modelo']
        melhor_pipe = melhores_estimadores[melhor_nome]
        
        # Avaliação
        pipe, thr, probs, y_pred = avaliar_teste(melhor_pipe, X_test, y_test, best_thr=0.4)
        
        f1_teste = f1_score(y_test, y_pred, zero_division=0)
        rec_teste = recall_score(y_test, y_pred, zero_division=0)
        prec_teste = precision_score(y_test, y_pred, zero_division=0)
        
        relatorio_final.append({
            'Abordagem': nome_ab,
            'Melhor Modelo': melhor_nome,
            'Features Usadas': len(cols_ativas),
            'PR_AUC (Treino CV)': df_cv.iloc[0]['PR_AUC'],
            'Recall (Teste)': rec_teste,
            'Precisão (Teste)': prec_teste,
            'F1 (Teste)': f1_teste
        })

        # --- PREPARAÇÃO DOS DADOS PARA O GRÁFICO FINAL ---
        resultados_para_plot[nome_ab] = {
            'y_test': y_test,
            'y_pred_test': y_pred
        }

        print(f"  Melhor Modelo: {melhor_nome}")
        print(f"  Relatório no Teste:")
        print(classification_report(y_test, y_pred, labels=[0, 1], zero_division=0))

# =======================================================
# EXIBIÇÃO DO RELATÓRIO E GRÁFICOS
# =======================================================
if relatorio_final:
    print("\n" + "="*80)
    print("RELATÓRIO COMPARATIVO DE DESEMPENHO NO TESTE".center(80))
    print("="*80)
    df_relatorio = pd.DataFrame(relatorio_final)
    # Formatação extra para o print ficar bonito no console
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df_relatorio.to_string(index=False))
    
    # Chama a função de plotagem (Ajustada para não precisar de df_cv_consolidado)
    plotar_resultados(resultados_para_plot, f"{MODO} ({TRATAMENTO_ALVO})", None)

## Crystallizer #2

In [ ]:
MODO = "C2"
TRATAMENTO_ALVO = "IQR" # Escolha o tratamento base (Original, IQR, etc.) para rodar os testes

print(f"\nIniciando testes comparativos no {MODO} usando tratamento {TRATAMENTO_ALVO}\n")

df_med, df_ev = CONFIGS[MODO][TRATAMENTO_ALVO]
df_feat, cols_feat_brutas = extrair_features(MODO, df_med, df_ev, JANELAS, STATS_FUNCS)

abordagens = [
    {"nome": "Base (Sem SMOTE, Sem FS)", "smote": False, "fs": False},
    {"nome": "Apenas SMOTE", "smote": True, "fs": False},
    {"nome": "Apenas Feature Selection", "smote": False, "fs": True},
    {"nome": "SMOTE + Feature Selection", "smote": True, "fs": True}
]

relatorio_final = []
resultados_para_plot = {} # <--- DICIONÁRIO CRIADO AQUI

if df_feat.empty or len(df_feat) < N_SPLITS + 2:
    print("Dados insuficientes.")
else:
    for abordagem in abordagens:
        nome_ab = abordagem['nome']
        print(f"\n{'='*55}")
        print(f"Executando: {nome_ab}")
        print(f"{'='*55}")

        X_train_b, X_test_b, y_train, y_test, _, _ = dividir_dados(df_feat, cols_feat_brutas, test_size=TEST_SIZE)
        
        # Lógica da Feature Selection
        if abordagem['fs']:
            cols_ativas = selecionar_features(X_train_b, y_train, cols_feat_brutas, k_features=10)
            if not cols_ativas: 
                print(f"  -> AVISO: Nenhuma feature sobreviveu à seleção em {nome_ab}. Pulando.")
                continue
        else:
            cols_ativas = cols_feat_brutas
            
        X_train, X_test = X_train_b[cols_ativas], X_test_b[cols_ativas]

        modelos_grids = definir_modelos_e_grids(y_train)
        df_cv, melhores_estimadores = avaliar_cv_com_grid(X_train, y_train, modelos_grids, usar_smote=abordagem['smote'])
        
        melhor_nome = df_cv.iloc[0]['Modelo']
        melhor_pipe = melhores_estimadores[melhor_nome]
        
        # Avaliação
        pipe, thr, probs, y_pred = avaliar_teste(melhor_pipe, X_test, y_test, best_thr=0.4)
        
        f1_teste = f1_score(y_test, y_pred, zero_division=0)
        rec_teste = recall_score(y_test, y_pred, zero_division=0)
        prec_teste = precision_score(y_test, y_pred, zero_division=0)
        
        relatorio_final.append({
            'Abordagem': nome_ab,
            'Melhor Modelo': melhor_nome,
            'Features Usadas': len(cols_ativas),
            'PR_AUC (Treino CV)': df_cv.iloc[0]['PR_AUC'],
            'Recall (Teste)': rec_teste,
            'Precisão (Teste)': prec_teste,
            'F1 (Teste)': f1_teste
        })

        # --- PREPARAÇÃO DOS DADOS PARA O GRÁFICO FINAL ---
        resultados_para_plot[nome_ab] = {
            'y_test': y_test,
            'y_pred_test': y_pred
        }

        print(f"  Melhor Modelo: {melhor_nome}")
        print(f"  Relatório no Teste:")
        print(classification_report(y_test, y_pred, labels=[0, 1], zero_division=0))

# =======================================================
# EXIBIÇÃO DO RELATÓRIO E GRÁFICOS
# =======================================================
if relatorio_final:
    print("\n" + "="*80)
    print("RELATÓRIO COMPARATIVO DE DESEMPENHO NO TESTE".center(80))
    print("="*80)
    df_relatorio = pd.DataFrame(relatorio_final)
    # Formatação extra para o print ficar bonito no console
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df_relatorio.to_string(index=False))
    
    # Chama a função de plotagem (Ajustada para não precisar de df_cv_consolidado)
    plotar_resultados(resultados_para_plot, f"{MODO} ({TRATAMENTO_ALVO})", None)

## Crystallizer #3

In [ ]:
MODO = "C3"
TRATAMENTO_ALVO = "IQR" # Escolha o tratamento base (Original, IQR, etc.) para rodar os testes

print(f"\nIniciando testes comparativos no {MODO} usando tratamento {TRATAMENTO_ALVO}\n")

df_med, df_ev = CONFIGS[MODO][TRATAMENTO_ALVO]
df_feat, cols_feat_brutas = extrair_features(MODO, df_med, df_ev, JANELAS, STATS_FUNCS)

abordagens = [
    {"nome": "Base (Sem SMOTE, Sem FS)", "smote": False, "fs": False},
    {"nome": "Apenas SMOTE", "smote": True, "fs": False},
    {"nome": "Apenas Feature Selection", "smote": False, "fs": True},
    {"nome": "SMOTE + Feature Selection", "smote": True, "fs": True}
]

relatorio_final = []
resultados_para_plot = {} # <--- DICIONÁRIO CRIADO AQUI

if df_feat.empty or len(df_feat) < N_SPLITS + 2:
    print("Dados insuficientes.")
else:
    for abordagem in abordagens:
        nome_ab = abordagem['nome']
        print(f"\n{'='*55}")
        print(f"Executando: {nome_ab}")
        print(f"{'='*55}")

        X_train_b, X_test_b, y_train, y_test, _, _ = dividir_dados(df_feat, cols_feat_brutas, test_size=TEST_SIZE)
        
        # Lógica da Feature Selection
        if abordagem['fs']:
            cols_ativas = selecionar_features(X_train_b, y_train, cols_feat_brutas, k_features=10)
            if not cols_ativas: 
                print(f"  -> AVISO: Nenhuma feature sobreviveu à seleção em {nome_ab}. Pulando.")
                continue
        else:
            cols_ativas = cols_feat_brutas
            
        X_train, X_test = X_train_b[cols_ativas], X_test_b[cols_ativas]

        modelos_grids = definir_modelos_e_grids(y_train)
        df_cv, melhores_estimadores = avaliar_cv_com_grid(X_train, y_train, modelos_grids, usar_smote=abordagem['smote'])
        
        melhor_nome = df_cv.iloc[0]['Modelo']
        melhor_pipe = melhores_estimadores[melhor_nome]
        
        # Avaliação
        pipe, thr, probs, y_pred = avaliar_teste(melhor_pipe, X_test, y_test, best_thr=0.4)
        
        f1_teste = f1_score(y_test, y_pred, zero_division=0)
        rec_teste = recall_score(y_test, y_pred, zero_division=0)
        prec_teste = precision_score(y_test, y_pred, zero_division=0)
        
        relatorio_final.append({
            'Abordagem': nome_ab,
            'Melhor Modelo': melhor_nome,
            'Features Usadas': len(cols_ativas),
            'PR_AUC (Treino CV)': df_cv.iloc[0]['PR_AUC'],
            'Recall (Teste)': rec_teste,
            'Precisão (Teste)': prec_teste,
            'F1 (Teste)': f1_teste
        })

        # --- PREPARAÇÃO DOS DADOS PARA O GRÁFICO FINAL ---
        resultados_para_plot[nome_ab] = {
            'y_test': y_test,
            'y_pred_test': y_pred
        }

        print(f"  Melhor Modelo: {melhor_nome}")
        print(f"  Relatório no Teste:")
        print(classification_report(y_test, y_pred, labels=[0, 1], zero_division=0))

# =======================================================
# EXIBIÇÃO DO RELATÓRIO E GRÁFICOS
# =======================================================
if relatorio_final:
    print("\n" + "="*80)
    print("RELATÓRIO COMPARATIVO DE DESEMPENHO NO TESTE".center(80))
    print("="*80)
    df_relatorio = pd.DataFrame(relatorio_final)
    # Formatação extra para o print ficar bonito no console
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df_relatorio.to_string(index=False))
    
    # Chama a função de plotagem (Ajustada para não precisar de df_cv_consolidado)
    plotar_resultados(resultados_para_plot, f"{MODO} ({TRATAMENTO_ALVO})", None)

## Crystallizer #1 #2 #3

In [ ]:
MODO = "Unificado"

# Dataframes unificados de medições — já construídos anteriormente
CONFIGS_UNIFICADO = {
    "Original":       df_crystallizer123,
    "Intervalo 0-10": df_crystallizer123_0a10,
    "IQR":            df_crystallizer123_iqr,
    "Hampel":         df_crystallizer123_hampel,
}

resultados_por_tratamento_unif = {}
df_cv_todos_unif = []

for tratamento, df_med_unif in CONFIGS_UNIFICADO.items():

    print(f"\n{'='*55}")
    print(f"{MODO} | {tratamento} (Absolutas + Relativas Misturadas)")
    print(f"{'='*55}")

    # Extração EXATAMENTE como você solicitou: base e eventos tratados como únicos.
    # O map_medicoes_unif é passado como None para forçar a mistura temporal.
    df_feat, cols_feat = extrair_features(
        crystallizer=MODO,
        df_medicoes=df_med_unif,               
        df_eventos=df_eventos_crystallizer123, 
        janelas=JANELAS, 
        stats_funcs=STATS_FUNCS,
        map_medicoes_unif=None                 
    )
    
    # Divisão sem label de origem do equipamento
    X_train, X_test, y_train, y_test, X_full, y_full = dividir_dados(
        df_feat, cols_feat, test_size=TEST_SIZE, incluir_crystallizer=False 
    )

    print(f"  Treino : {X_train.shape[0]} amostras "
          f"(Real=1: {y_train.sum()}  Real=0: {(y_train==0).sum()})")
    print(f"  Teste  : {X_test.shape[0]} amostras  "
          f"(Real=1: {y_test.sum()}  Real=0: {(y_test==0).sum()})")

    # Validação cruzada com GridSearch no treino
    modelos_grids = definir_modelos_e_grids(y_train)
    df_cv, melhores_estimadores = avaliar_cv_com_grid(X_train, y_train, modelos_grids)
    df_cv.insert(0, 'Tratamento', tratamento)
    df_cv_todos_unif.append(df_cv)

    print("\n  CV (treino):")
    print(df_cv[['Modelo','F1','PR_AUC','ROC_AUC','Recall']].to_string(index=False))

    # Teste final usando o melhor estimador do GridSearch
    melhor_nome = df_cv.iloc[0]['Modelo']
    melhor_pipe = melhores_estimadores[melhor_nome]

    # Avaliação com o threshold que definimos na função (0.35)
    pipe, thr, probs_test, y_pred_test = avaliar_teste(
        melhor_pipe, X_train, y_train, X_test, y_test
    )

    resultados_por_tratamento_unif[tratamento] = {
        'pipe':       pipe,
        'threshold':  thr,
        'probs_test': probs_test,
        'y_test':     y_test,
        'y_pred_test': y_pred_test,
        'melhor':     melhor_nome,
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
    }

    print(f"\n  Melhor modelo : {melhor_nome}  |  threshold: {thr:.3f}")
    print(f"\n  Relatório no Teste:")
    print(classification_report(
        y_test, y_pred_test,
        labels=[0, 1],
        target_names=['Falso Positivo', 'Contaminação Real'], 
        zero_division=0
    ))

df_cv_consolidado_unif = pd.concat(df_cv_todos_unif, ignore_index=True)
plotar_resultados(resultados_por_tratamento_unif, MODO, df_cv_consolidado_unif)

# IsolationForest

In [ ]:
JANELAS = [6, 3, 1]
N_SPLITS     = 5
RANDOM_STATE = 42
TEST_SIZE    = 0.1   # 20% para teste
VAL_SIZE     = 0.2   # 20% do treino para validação (dentro do CV)

STATS_FUNCS = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max
}

# Definindo o grid de parâmetros para testar
PARAM_GRID = {
    'contamination': [0.01, 0.03, 0.05, 0.10],
    'n_estimators': [100, 300, 500],
    'max_features': [0.5, 0.8, 1.0],
    'max_samples': ['auto', 0.5, 0.8]
}

def otimizar_avaliar_iforest(X_train, y_train, X_test, y_test, param_grid):
    """
    Testa várias combinações de hiperparâmetros usando as anomalias do CONJUNTO DE TREINO 
    para escolher o melhor modelo, mantendo o conjunto de teste fora.
    """
    X_train_normal = X_train[y_train == 0]
    
    melhor_score = -1
    melhores_params = {}
    melhor_pipe = None
    
    # Gera todas as combinações possíveis
    chaves = param_grid.keys()
    combinacoes = list(itertools.product(*param_grid.values())) # Transformado em lista por segurança
    
    for config in combinacoes:
        params = dict(zip(chaves, config))
        
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('iforest', IsolationForest(
                n_estimators=params['n_estimators'], 
                max_samples=params['max_samples'],
                max_features=params['max_features'],
                contamination=params['contamination'], 
                random_state=RANDOM_STATE, 
                n_jobs=-1
            ))
        ])
        
        # Treina estritamente nos dados normais do passado
        pipe.fit(X_train_normal)
        
        # AVALIAÇÃO NO TREINO: O modelo tenta prever o próprio treino (que contém anomalias que ele não viu no fit)
        preds_train = pipe.predict(X_train)
        y_pred_train = np.where(preds_train == -1, 1, 0)
        
        # O score que define o melhor modelo é o do treino
        score_train = fbeta_score(y_train, y_pred_train, beta=2, pos_label=1, zero_division=0)
        
        if score_train > melhor_score:
            melhor_score = score_train
            melhores_params = params
            melhor_pipe = pipe
            
    # APÓS escolher o melhor modelo é feita uma ÚNICA predição no conjunto de Teste
    preds_test = melhor_pipe.predict(X_test)
    melhor_y_pred_test = np.where(preds_test == -1, 1, 0)
            
    return melhor_pipe, melhor_y_pred_test, melhores_params, melhor_score

def plotar_resultados_iforest(y_test, y_pred, tratamento_nome):
    """Gera apenas a Matriz de Confusão."""
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=['Prev: Normal', 'Prev: Anomalia'],
        y=['Real: Normal', 'Real: Contaminação'],
        colorscale='Blues',
        text=cm,
        texttemplate="%{text}",
        showscale=False
    ))

    fig.update_layout(
        title=f"Matriz de Confusão — Isolation Forest — {tratamento_nome}",
        template="plotly_white",
        height=400,
        width=500
    )
    
    fig.show()

## Crystallizer #1

In [ ]:
MODO = "C1"

for tratamento, (df_med, df_ev) in CONFIGS[MODO].items():

    print(f"\n{'='*55}")
    print(f"{MODO} | {tratamento} (Isolation Forest - Otimizado)")
    print(f"{'='*55}")

    df_feat, cols_feat = extrair_features(
        MODO, df_med, df_ev,
        JANELAS, STATS_FUNCS
    )

    # Feature de Salto Abrupto (Variação Percentual)
    if 'max_1d' in df_feat.columns and 'media_6d' in df_feat.columns:
        df_feat['salto_abrupto'] = (df_feat['max_1d'] - df_feat['media_6d']) / (df_feat['media_6d'] + 0.001)
        cols_feat.append('salto_abrupto')
    
    # Divisão temporal 
    X_train, X_test, y_train, y_test, _, _ = dividir_dados(
        df_feat, cols_feat, incluir_crystallizer=False
    )

    print(f"  Treino : {X_train.shape[0]} amostras (usando apenas as normais)")
    print(f"  Teste  : {X_test.shape[0]} amostras  (Real=1: {y_test.sum()}  Real=0: {(y_test==0).sum()})")

    # Chama a nova função de otimização
    pipe_if, y_pred_if, melhores_params, melhor_score = otimizar_avaliar_iforest(
        X_train, y_train, X_test, y_test, param_grid=PARAM_GRID
    )

    print(f"\n  Melhores Parâmetros Encontrados:")
    for k, v in melhores_params.items():
        print(f"  - {k}: {v}")
    print(f"  - F2-Score: {melhor_score:.4f}")

    print(f"\n  Relatório no Teste:")
    print(classification_report(
        y_test, y_pred_if,
        labels=[0, 1],
        target_names=['Operação Normal', 'Contaminação Detectada'],
        zero_division=0
    ))
    
    plotar_resultados_iforest(y_test, y_pred_if, tratamento)

## Crystallizer #2

In [ ]:
MODO = "C2"

for tratamento, (df_med, df_ev) in CONFIGS[MODO].items():

    print(f"\n{'='*55}")
    print(f"{MODO} | {tratamento} (Isolation Forest - Otimizado)")
    print(f"{'='*55}")

    df_feat, cols_feat = extrair_features(
        MODO, df_med, df_ev,
        JANELAS, STATS_FUNCS
    )

    # Feature de Salto Abrupto (Variação Percentual)
    if 'max_1d' in df_feat.columns and 'media_6d' in df_feat.columns:
        df_feat['salto_abrupto'] = (df_feat['max_1d'] - df_feat['media_6d']) / (df_feat['media_6d'] + 0.001)
        cols_feat.append('salto_abrupto')
    
    # Divisão temporal 
    X_train, X_test, y_train, y_test, _, _ = dividir_dados(
        df_feat, cols_feat, incluir_crystallizer=False
    )

    print(f"  Treino : {X_train.shape[0]} amostras (usando apenas as normais)")
    print(f"  Teste  : {X_test.shape[0]} amostras  (Real=1: {y_test.sum()}  Real=0: {(y_test==0).sum()})")

    # Chama a nova função de otimização
    pipe_if, y_pred_if, melhores_params, melhor_score = otimizar_avaliar_iforest(
        X_train, y_train, X_test, y_test, param_grid=PARAM_GRID
    )

    print(f"\n  Melhores Parâmetros Encontrados:")
    for k, v in melhores_params.items():
        print(f"  - {k}: {v}")
    print(f"  - F2-Score: {melhor_score:.4f}")

    print(f"\n  Relatório no Teste:")
    print(classification_report(
        y_test, y_pred_if,
        labels=[0, 1],
        target_names=['Operação Normal', 'Contaminação Detectada'], 
        zero_division=0
    ))
    
    plotar_resultados_iforest(y_test, y_pred_if, tratamento)

## Crystallizer #3

In [ ]:
MODO = "C3"

for tratamento, (df_med, df_ev) in CONFIGS[MODO].items():

    print(f"\n{'='*55}")
    print(f"{MODO} | {tratamento} (Isolation Forest - Otimizado)")
    print(f"{'='*55}")

    df_feat, cols_feat = extrair_features(
        MODO, df_med, df_ev,
        JANELAS, STATS_FUNCS
    )

    # Feature de Salto Abrupto (Variação Percentual)
    if 'max_1d' in df_feat.columns and 'media_6d' in df_feat.columns:
        df_feat['salto_abrupto'] = (df_feat['max_1d'] - df_feat['media_6d']) / (df_feat['media_6d'] + 0.001)
        cols_feat.append('salto_abrupto')
    
    # Divisão temporal 
    X_train, X_test, y_train, y_test, _, _ = dividir_dados(
        df_feat, cols_feat, incluir_crystallizer=False
    )

    print(f"  Treino : {X_train.shape[0]} amostras (usando apenas as normais)")
    print(f"  Teste  : {X_test.shape[0]} amostras  (Real=1: {y_test.sum()}  Real=0: {(y_test==0).sum()})")

    # Chama a nova função de otimização
    pipe_if, y_pred_if, melhores_params, melhor_score = otimizar_avaliar_iforest(
        X_train, y_train, X_test, y_test, param_grid=PARAM_GRID
    )

    print(f"\n  Melhores Parâmetros Encontrados:")
    for k, v in melhores_params.items():
        print(f"  - {k}: {v}")
    print(f"  - F2-Score: {melhor_score:.4f}")

    print(f"\n  Relatório no Teste:")
    print(classification_report(
        y_test, y_pred_if,
        labels=[0, 1],
        target_names=['Operação Normal', 'Contaminação Detectada'], 
        zero_division=0
    ))
    
    plotar_resultados_iforest(y_test, y_pred_if, tratamento)

## Crystallizer #1#2#3

In [ ]:
MODO = "Unificado"

for tratamento, df_med_unif in CONFIGS_UNIFICADO.items():

    print(f"\n{'='*55}")
    print(f"{MODO} | {tratamento} (Isolation Forest - Otimizado)")
    print(f"{'='*55}")

    # No unificado, o df_med vem do dicionário e o df_ev é fixo
    df_med = df_med_unif
    df_ev  = df_eventos_crystallizer123

    df_feat, cols_feat = extrair_features(
        MODO, df_med, df_ev,
        JANELAS, STATS_FUNCS
    )

    # Feature de Salto Abrupto (Variação Percentual)
    if 'max_1d' in df_feat.columns and 'media_6d' in df_feat.columns:
        df_feat['salto_abrupto'] = (df_feat['max_1d'] - df_feat['media_6d']) / (df_feat['media_6d'] + 0.001)
        cols_feat.append('salto_abrupto')
    
    # Divisão temporal 
    X_train, X_test, y_train, y_test, _, _ = dividir_dados(
        df_feat, cols_feat, incluir_crystallizer=False
    )

    print(f"  Treino : {X_train.shape[0]} amostras (usando apenas as normais)")
    print(f"  Teste  : {X_test.shape[0]} amostras  (Real=1: {y_test.sum()}  Real=0: {(y_test==0).sum()})")

    # Chama a função de otimização (Grid Search Manual)
    pipe_if, y_pred_if, melhores_params, melhor_score = otimizar_avaliar_iforest(
        X_train, y_train, X_test, y_test, param_grid=PARAM_GRID
    )

    print(f"\n  Melhores Parâmetros Encontrados:")
    for k, v in melhores_params.items():
        print(f"  - {k}: {v}")
    print(f"  - F2-Score: {melhor_score:.4f}")

    print(f"\n  Relatório no Teste:")
    print(classification_report(
        y_test, y_pred_if,
        labels=[0, 1],
        target_names=['Operação Normal', 'Contaminação Detectada'], 
        zero_division=0
    ))
    
    plotar_resultados_iforest(y_test, y_pred_if, tratamento)

# Carta de Controle
Avaliando carta de controle EWMA verificando se há antecedência nos eventos de falha no reator

In [ ]:
def calcular_ewma(serie, media_baseline, std_baseline, lambd=0.2, L=3):
    """
    Calcula a carta EWMA com baseline fixo.
    
    lambd : fator de suavização (0 < λ ≤ 1) — menor = mais suave
    L     : multiplicador do limite de controle — menor = mais sensível
    """
    n = len(serie)
    ewma = np.zeros(n)
    ucl  = np.zeros(n)
    
    ewma[0] = media_baseline

    for i in range(1, n):
        ewma[i] = lambd * serie.iloc[i] + (1 - lambd) * ewma[i - 1]
        # Variância assintótica da EWMA
        sigma_ewma = std_baseline * np.sqrt(lambd / (2 - lambd) * (1 - (1 - lambd) ** (2 * (i + 1))))
        ucl[i] = media_baseline + L * sigma_ewma

    alarmes = ewma > ucl

    df_ewma = pd.DataFrame({
        'Valor'  : serie.values,
        'EWMA'   : ewma,
        'UCL'    : ucl,
        'Alarme' : alarmes
    }, index=serie.index)

    return df_ewma


def otimizar_ewma(serie, df_eventos, media_hist, std_hist, param_grid):
    """
    Busca os melhores hiperparâmetros para a carta EWMA baseando-se no maior F2-Score.
    """
    melhor_f2 = -1
    melhores_params = {}
    
    chaves = param_grid.keys()
    combinacoes = list(itertools.product(*param_grid.values()))
    
    print(f"Iniciando otimização com {len(combinacoes)} combinações...")
    
    for config in combinacoes:
        params = dict(zip(chaves, config))
        
        # 1. Gera a carta de controle com os parâmetros atuais
        df_ewma_temp = calcular_ewma(
            serie, media_hist, std_hist, 
            lambd=params['lambd'], L=params['L']
        )
        
        # 2. Avalia a carta usando a janela de dias atual
        y_true = []
        y_pred = []
        janela = params['janela_dias']
        
        for _, evento in df_eventos.iterrows():
            ts = evento["TIMESTAMP"]
            classe = int(evento["Real"])
            
            inicio = ts - pd.Timedelta(days=janela)
            mask = (df_ewma_temp.index >= inicio) & (df_ewma_temp.index < ts)
            slice_controle = df_ewma_temp[mask]
            
            teve_alarme = 1 if slice_controle['Alarme'].any() else 0
            
            y_true.append(classe)
            y_pred.append(teve_alarme)
            
        # 3. Calcula F2
        f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        
        # 4. Atualiza os melhores parâmetros
        if f2 > melhor_f2:
            melhor_f2 = f2
            melhores_params = params
            
    print(f"\nMelhor F2-Score encontrado: {melhor_f2:.4f}")
    print(f"Melhores parâmetros: {melhores_params}")
    
    return melhores_params, melhor_f2


def plotar_carta_ewma(df_ewma, df_eventos, titulo="Carta EWMA", mostrar_falsos=True):
    """Visualização da carta EWMA com a lógica de shapes para as linhas de eventos."""
    fig = go.Figure()

    # Estatística de Controle (EWMA)
    fig.add_trace(go.Scatter(x=df_ewma.index, y=df_ewma['EWMA'], 
                             mode='lines', name='EWMA', line=dict(color='black')))
    
    # Limite de Controle (UCL)
    fig.add_trace(go.Scatter(x=df_ewma.index, y=df_ewma['UCL'], 
                             mode='lines', name='Limite (UCL)', 
                             line=dict(color='red', dash='dash')))
    
    # Pontos de Alarme
    alarmes = df_ewma[df_ewma['Alarme']]
    fig.add_trace(go.Scatter(x=alarmes.index, y=alarmes['EWMA'], 
                             mode='markers', name='Alarme EWMA', 
                             marker=dict(color='red', size=8, symbol='x')))

    # Adição dos eventos com a mesma lógica do seu plot_crystallizer
    for _, row in df_eventos.iterrows():
        if not mostrar_falsos and row["Real"] == 0:
            continue

        cor = "red" if row["Real"] == 1 else "blue"

        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0, y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )

    fig.update_layout(
        title=titulo, 
        template="plotly_white", 
        height=450,
        hovermode='x unified'
    )
    fig.show()


def avaliar_carta_controle(df_controle, df_eventos, nome_carta, janela_dias=3):
    """
    Avalia a carta de controle verificando se ela disparou na janela de tempo antes do evento.
    """
    y_true = []
    y_pred = []
    
    for _, evento in df_eventos.iterrows():
        ts = evento["TIMESTAMP"]
        classe = int(evento["Real"])
        
        # Define a janela de busca para o alarme (ex: últimos 3 dias antes do evento)
        inicio = ts - pd.Timedelta(days=janela_dias)
        
        # Filtra a carta de controle nessa janela
        mask = (df_controle.index >= inicio) & (df_controle.index < ts)
        slice_controle = df_controle[mask]
        
        # Se houve QUALQUER alarme nessa janela, prevemos 1 (Anomalia)
        teve_alarme = 1 if slice_controle['Alarme'].any() else 0
        
        y_true.append(classe)
        y_pred.append(teve_alarme)
        
    f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    
    print(f"\n{'='*55}")
    print(f"Avaliação da Carta: {nome_carta} (Janela de {janela_dias} dias)")
    print(f"{'='*55}")
    print(f"F2-Score: {f2:.4f}")
    print("\nRelatório de Classificação:")
    print(classification_report(y_true, y_pred, labels=[0, 1], 
                                target_names=['Operação Normal', 'Contaminação Detectada'], 
                                zero_division=0))
    return y_true, y_pred, f2

## EWMA

In [ ]:
df_c1 = df_crystallizer1.copy()
df_c1['TIMESTAMP'] = pd.to_datetime(df_c1['TIMESTAMP'])
df_c1 = df_c1.sort_values('TIMESTAMP').set_index('TIMESTAMP')
serie_fe = df_c1['Resultado de Ferro (ppm)'].dropna()

df_eventos_c1 = df_eventos_crystallizer1.copy()
df_eventos_c1['TIMESTAMP'] = pd.to_datetime(df_eventos_c1['TIMESTAMP'])

periodos_baseline = [
    ('2012-03-01', '2013-03-25'),
    ('2014-01-01', '2015-01-01'),
    # ('2022-05-01', '2023-10-01')
]
fatias = [serie_fe.loc[inicio:fim] for inicio, fim in periodos_baseline]
serie_baseline = pd.concat(fatias)

media_historica = serie_baseline.mean()
std_historico   = serie_baseline.std()

print(f"Baseline | Média: {media_historica:.3f} ppm | Std: {std_historico:.3f} ppm")
print(f"Total de amostras no baseline: {len(serie_baseline)}\n")

# 1. Defina as opções matemáticas que você quer testar
param_grid_ewma = {
    'lambd': [0.1, 0.2, 0.3, 0.4, 0.5],       # O peso dos dados recentes
    'L': [3, 5, 7, 9, 11, 13],                # O multiplicador do limite
    'janela_dias': [3, 6, 9, 12, 15]          # Antecedência do alarme
}

# 2. Roda a otimização
melhores_parametros, max_f2 = otimizar_ewma(
    serie_fe, df_eventos_c1, media_historica, std_historico, param_grid_ewma
)

# 3. Gera a carta final com a melhor configuração encontrada
df_ewma_otimizado = calcular_ewma(
    serie_fe, media_historica, std_historico, 
    lambd=melhores_parametros['lambd'], 
    L=melhores_parametros['L']
)

# 4. Avalia formalmente para imprimir o relatório completo
avaliar_carta_controle(
    df_ewma_otimizado, df_eventos_c1, 
    nome_carta=f"EWMA Otimizado (λ={melhores_parametros['lambd']}, L={melhores_parametros['L']})", 
    janela_dias=melhores_parametros['janela_dias']
)

# 5. Plota o gráfico final
plotar_carta_ewma(
    df_ewma_otimizado, df_eventos_c1,
    titulo=f"Carta EWMA - C1 (λ={melhores_parametros['lambd']}, L={melhores_parametros['L']})",
    mostrar_falsos=False
)

## CUMSUM

In [ ]:
def calcular_cusum_dinamico(serie, janela_baseline=30, k=0.5, h=4):
    """
    Calcula o CUSUM adaptativo. A média e desvio padrão são atualizados 
    continuamente usando uma janela móvel do passado.
    """
    n = len(serie)
    c_pos = np.zeros(n)
    limite_h = np.zeros(n)
    alarmes = np.zeros(n, dtype=bool)
    
    # Média e std móveis. O shift(1) garante que o valor de "hoje" não contamine o baseline de "hoje"
    media_movel = serie.shift(1).rolling(window=janela_baseline, min_periods=janela_baseline//2).mean()
    std_movel = serie.shift(1).rolling(window=janela_baseline, min_periods=janela_baseline//2).std()
    
    for i in range(1, n):
        mu = media_movel.iloc[i]
        sigma = std_movel.iloc[i]
        
        # Pula se não houver histórico suficiente ou se o desvio for zero absoluto
        if pd.isna(mu) or pd.isna(sigma) or sigma == 0:
            continue
            
        # Calcula os limites dinâmicos para o dia 'i'
        K_val = k * sigma
        H_val = h * sigma
        limite_h[i] = H_val
        
        # Acumula o desvio positivo
        c_pos[i] = max(0, c_pos[i-1] + serie.iloc[i] - mu - K_val)
        
        # Dispara o alarme e reseta a soma (crucial para evitar fadiga de alarme)
        if c_pos[i] > H_val:
            alarmes[i] = True
            c_pos[i] = 0 
            
    df_cusum = pd.DataFrame({
        'Valor': serie.values,
        'CUSUM_Positivo': c_pos,
        'Limite_H': limite_h,
        'Alarme': alarmes
    }, index=serie.index)
    
    return df_cusum

def avaliar_carta_controle(df_controle, df_eventos, nome_carta, janela_dias=3):
    """Avalia a precisão dos alarmes em relação aos eventos reais."""
    y_true = []
    y_pred = []
    
    for _, evento in df_eventos.iterrows():
        ts = evento["TIMESTAMP"]
        classe = int(evento["Real"])
        
        inicio = ts - pd.Timedelta(days=janela_dias)
        mask = (df_controle.index >= inicio) & (df_controle.index < ts)
        slice_controle = df_controle[mask]
        
        teve_alarme = 1 if slice_controle['Alarme'].any() else 0
        
        y_true.append(classe)
        y_pred.append(teve_alarme)
        
    f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    
    print(f"\n{'='*55}")
    print(f"Avaliação da Carta: {nome_carta} (Janela de {janela_dias} dias)")
    print(f"{'='*55}")
    print(f"F2-Score: {f2:.4f}")
    print("\nRelatório de Classificação:")
    print(classification_report(y_true, y_pred, labels=[0, 1], 
                                target_names=['Operação Normal', 'Contaminação Detectada'], 
                                zero_division=0))
    return y_true, y_pred, f2

def otimizar_cusum_dinamico(serie, df_eventos, param_grid):
    """Grid Search manual para encontrar a melhor parametrização do CUSUM Dinâmico."""
    melhor_f2 = -1
    melhores_params = {}
    
    chaves = param_grid.keys()
    combinacoes = list(itertools.product(*param_grid.values()))
    
    print(f"Iniciando otimização do CUSUM Dinâmico com {len(combinacoes)} combinações...")
    
    for config in combinacoes:
        params = dict(zip(chaves, config))
        
        df_cusum_temp = calcular_cusum_dinamico(
            serie, 
            janela_baseline=params['janela_baseline'], 
            k=params['k'], 
            h=params['h']
        )
        
        y_true = []
        y_pred = []
        janela = params['janela_dias']
        
        for _, evento in df_eventos.iterrows():
            ts = evento["TIMESTAMP"]
            classe = int(evento["Real"])
            
            inicio = ts - pd.Timedelta(days=janela)
            mask = (df_cusum_temp.index >= inicio) & (df_cusum_temp.index < ts)
            slice_controle = df_cusum_temp[mask]
            
            teve_alarme = 1 if slice_controle['Alarme'].any() else 0
            
            y_true.append(classe)
            y_pred.append(teve_alarme)
            
        f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        
        if f2 > melhor_f2:
            melhor_f2 = f2
            melhores_params = params
            
    print(f"\nMelhor F2-Score encontrado: {melhor_f2:.4f}")
    print(f"Melhores parâmetros: {melhores_params}")
    
    return melhores_params, melhor_f2

# ==========================================
# 2. FUNÇÃO DE PLOTAGEM
# ==========================================

def plotar_carta_cusum(df_cusum, df_eventos, titulo="Carta CUSUM Dinâmico", mostrar_falsos=True):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=df_cusum.index, y=df_cusum['CUSUM_Positivo'], 
                             mode='lines', name='CUSUM Positivo', line=dict(color='black')))
    
    fig.add_trace(go.Scatter(x=df_cusum.index, y=df_cusum['Limite_H'], 
                             mode='lines', name='Limite Dinâmico (H)', 
                             line=dict(color='red', dash='dash')))
    
    alarmes = df_cusum[df_cusum['Alarme']]
    fig.add_trace(go.Scatter(x=alarmes.index, y=alarmes['CUSUM_Positivo'], 
                             mode='markers', name='Alarme', 
                             marker=dict(color='red', size=8, symbol='x')))

    for _, row in df_eventos.iterrows():
        if not mostrar_falsos and row["Real"] == 0:
            continue

        cor = "red" if row["Real"] == 1 else "blue"
        texto = row["EVENTO"] if "EVENTO" in df_eventos.columns else ("Contaminação Real" if row["Real"] == 1 else "Evento Normal")

        fig.add_shape(
            type="line", x0=str(row["TIMESTAMP"]), x1=str(row["TIMESTAMP"]),
            y0=0, y1=1, yref="paper", line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]), y=1, yref="paper", text=texto,
            showarrow=False, textangle=-90, yanchor="top", font=dict(color=cor)
        )

    fig.update_layout(title=titulo, template="plotly_white", height=450, hovermode='x unified')
    fig.show()

# ==========================================
# 3. BLOCO DE EXECUÇÃO E OTIMIZAÇÃO
# ==========================================

# Preparação dos dados
df_c1 = df_crystallizer1.copy()
df_c1['TIMESTAMP'] = pd.to_datetime(df_c1['TIMESTAMP'])
df_c1 = df_c1.sort_values('TIMESTAMP').set_index('TIMESTAMP')
serie_fe = df_c1['Resultado de Ferro (ppm)'].dropna()

df_eventos_c1 = df_eventos_crystallizer1.copy()
df_eventos_c1['TIMESTAMP'] = pd.to_datetime(df_eventos_c1['TIMESTAMP'])

# Definição do Grid de Parâmetros para buscar a melhor performance
param_grid_cusum = {
    'janela_baseline': [15, 30, 45, 60], # Quantos dias passados compõem o "normal"
    'k': [0.25, 0.5, 0.75],              # Folga (menor = soma desvios menores)
    'h': [3, 4, 5, 6],                   # Limite de alarme (maior = mais rigoroso)
    'janela_dias': [3, 6, 9, 12]         # Janela de antecedência para prever a falha
}

# Roda a otimização
melhores_parametros, max_f2 = otimizar_cusum_dinamico(serie_fe, df_eventos_c1, param_grid_cusum)

# Gera a carta final com os hiperparâmetros campeões
df_cusum_otimizado = calcular_cusum_dinamico(
    serie_fe, 
    janela_baseline=melhores_parametros['janela_baseline'], 
    k=melhores_parametros['k'], 
    h=melhores_parametros['h']
)

# Avalia formalmente para exibir o relatório
nome_modelo = f"CUSUM Dinâmico (jan_base={melhores_parametros['janela_baseline']}, k={melhores_parametros['k']}, h={melhores_parametros['h']})"
avaliar_carta_controle(df_cusum_otimizado, df_eventos_c1, nome_carta=nome_modelo, janela_dias=melhores_parametros['janela_dias'])

# Plota o gráfico para análise visual
plotar_carta_cusum(
    df_cusum_otimizado, df_eventos_c1,
    titulo=f"Carta CUSUM Dinâmico - C1 {nome_modelo}",
    mostrar_falsos=False
)